In [ ]:
from pathlib import Path
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", Path.cwd())

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch check failed:", repr(exc))

print("\nKaggle input folders:")
for path in sorted(Path("/kaggle/input").iterdir()):
    print(" -", path)

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("Top-level Kaggle input folders:")
for path in sorted(input_root.iterdir()):
    print(" -", path)

print("\nSearching for HAM10000_metadata.csv...")
metadata_candidates = list(input_root.rglob("HAM10000_metadata.csv"))

for candidate in metadata_candidates:
    print("Metadata candidate:", candidate)

print("\nSearching for image folders...")
image_folder_candidates = []
for folder in input_root.rglob("*"):
    if folder.is_dir() and "HAM10000_images" in folder.name:
        jpg_count = len(list(folder.glob("*.jpg")))
        png_count = len(list(folder.glob("*.png")))
        image_folder_candidates.append((folder, jpg_count, png_count))

for folder, jpg_count, png_count in image_folder_candidates:
    print(f"Image folder: {folder} | jpg={jpg_count}, png={png_count}")

if not metadata_candidates:
    raise FileNotFoundError("HAM10000_metadata.csv was not found. Check whether the dataset was attached.")

print("\nDataset discovery completed.")

### 1st Training session 

In [ ]:
# Phase 1: Clone public GitHub repository

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/DiloJan-26/skin-lesion-uncertainty.git"
WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "skin-lesion-uncertainty"

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Current directory:", Path.cwd())

print("\nRepository files:")
for path in sorted(REPO_DIR.iterdir()):
    print(" -", path.name)

In [ ]:
# Phase 1: Install and verify extra dependencies

import subprocess
import sys

packages = [
    "timm>=0.9",
    "torchmetrics>=1.3",
    "PyYAML>=6.0"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

print("Extra dependencies installed.")

In [ ]:
# Phase 1: Repository config check

from src.config import load_config
from src.paths import ensure_output_dirs

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

print("Config loaded successfully.")
print("Project:", config["project"]["title"])
print("Classes:", config["data"]["classes"])

print("\nOutput directories:")
for name, path in output_dirs.items():
    print(f"{name}: {path}")

In [ ]:
# Phase 1: Final dataset validation and discovery summary

from pathlib import Path
import json
import pandas as pd

DATASET_ROOT = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")

metadata_path = DATASET_ROOT / "HAM10000_metadata.csv"
image_dirs = [
    DATASET_ROOT / "HAM10000_images_part_1",
    DATASET_ROOT / "HAM10000_images_part_2",
]

print("Dataset root:", DATASET_ROOT)
print("Metadata exists:", metadata_path.exists())

for image_dir in image_dirs:
    print(f"{image_dir.name} exists:", image_dir.exists())
    print(f"{image_dir.name} JPG count:", len(list(image_dir.glob("*.jpg"))))

metadata = pd.read_csv(metadata_path)

print("\nMetadata shape:", metadata.shape)
print("Columns:", metadata.columns.tolist())

required_columns = ["lesion_id", "image_id", "dx"]
missing_columns = [col for col in required_columns if col not in metadata.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nClass distribution:")
class_distribution = metadata["dx"].value_counts().sort_index()
display(class_distribution.rename_axis("class").reset_index(name="count"))

all_image_paths = []
for image_dir in image_dirs:
    all_image_paths.extend(list(image_dir.glob("*.jpg")))

image_path_map = {path.stem: path for path in all_image_paths}

metadata_image_ids = set(metadata["image_id"].astype(str))
image_file_ids = set(image_path_map.keys())

matched_ids = metadata_image_ids.intersection(image_file_ids)
missing_ids = metadata_image_ids.difference(image_file_ids)

print("\nImage mapping check:")
print("Total image files found:", len(image_path_map))
print("Metadata image IDs:", len(metadata_image_ids))
print("Matched image IDs:", len(matched_ids))
print("Missing image IDs:", len(missing_ids))

if missing_ids:
    print("First missing IDs:", sorted(list(missing_ids))[:10])
else:
    print("All metadata image IDs have matching image files.")

print("\nUnique lesion IDs:", metadata["lesion_id"].nunique())
print("Unique image IDs:", metadata["image_id"].nunique())
print("Total rows:", len(metadata))

summary = {
    "dataset_root": str(DATASET_ROOT),
    "metadata_path": str(metadata_path),
    "metadata_shape": list(metadata.shape),
    "columns": metadata.columns.tolist(),
    "class_distribution": class_distribution.to_dict(),
    "unique_lesion_ids": int(metadata["lesion_id"].nunique()),
    "unique_image_ids": int(metadata["image_id"].nunique()),
    "total_rows": int(len(metadata)),
    "image_folders": [
        {
            "path": str(image_dir),
            "jpg_count": int(len(list(image_dir.glob("*.jpg")))),
        }
        for image_dir in image_dirs
    ],
    "total_image_files_found": int(len(image_path_map)),
    "matched_image_ids": int(len(matched_ids)),
    "missing_image_ids": int(len(missing_ids)),
}

output_path = Path("outputs/tables/dataset_discovery_summary.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved dataset summary to:", output_path)

In [ ]:
# Phase 2: Update repository from GitHub

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull"], check=True)

print("Updated repository.")
print("Current directory:", Path.cwd())

In [ ]:
# Phase 2: Verify updated source imports

from src.data import (
    find_ham10000_paths,
    load_ham10000_metadata,
    build_image_path_map,
    attach_image_paths,
    create_grouped_stratified_split,
    verify_no_group_leakage,
    summarize_split,
    save_split_files,
)

from src.plots import (
    plot_overall_class_distribution,
    plot_class_distribution_by_split,
)

print("Phase 2 imports successful.")

In [ ]:
# Phase 2: Create leakage-safe HAM10000 train/validation/test split

from pathlib import Path
import pandas as pd

from src.config import load_config
from src.paths import ensure_output_dirs
from src.data import (
    find_ham10000_paths,
    load_ham10000_metadata,
    build_image_path_map,
    attach_image_paths,
    create_grouped_stratified_split,
    verify_no_group_leakage,
    summarize_split,
    save_split_files,
)
from src.plots import (
    plot_overall_class_distribution,
    plot_class_distribution_by_split,
)

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

paths = find_ham10000_paths("/kaggle/input")

print("Dataset root:", paths["dataset_root"])
print("Metadata path:", paths["metadata_path"])
print("Image directories:")
for image_dir in paths["image_dirs"]:
    print(" -", image_dir)

metadata = load_ham10000_metadata(paths["metadata_path"])
image_path_map = build_image_path_map(paths["image_dirs"])
metadata_with_paths = attach_image_paths(
    metadata=metadata,
    image_path_map=image_path_map,
    image_id_column=config["data"]["image_id_column"],
)

print("\nMetadata with image paths:", metadata_with_paths.shape)
print("Columns:", metadata_with_paths.columns.tolist())
print("Image path sample:")
display(metadata_with_paths[["image_id", "dx", "lesion_id", "image_path"]].head())

split_df = create_grouped_stratified_split(
    metadata=metadata_with_paths,
    label_column=config["data"]["label_column"],
    group_column=config["data"]["lesion_group_column"],
    train_fraction=config["data"]["train_fraction"],
    val_fraction=config["data"]["val_fraction"],
    test_fraction=config["data"]["test_fraction"],
    seed=config["training"]["ensemble_seeds"][0],
)

verify_no_group_leakage(
    split_df,
    group_column=config["data"]["lesion_group_column"],
    split_column="split",
)

split_summary, class_distribution_by_split = summarize_split(
    split_df,
    label_column=config["data"]["label_column"],
    split_column="split",
)

saved_paths = save_split_files(
    split_df=split_df,
    output_dir=output_dirs["tables_dir"],
    label_column=config["data"]["label_column"],
)

overall_plot_path = plot_overall_class_distribution(
    metadata=split_df,
    label_column=config["data"]["label_column"],
    output_path=output_dirs["figures_dir"] / "overall_class_distribution.png",
)

split_plot_path = plot_class_distribution_by_split(
    class_distribution_by_split=class_distribution_by_split,
    output_path=output_dirs["figures_dir"] / "class_distribution_by_split.png",
)

print("\nSplit summary:")
display(split_summary)

print("\nClass distribution by split:")
display(class_distribution_by_split)

print("\nSaved split files:")
for name, path in saved_paths.items():
    print(f"{name}: {path}")

print("\nSaved plots:")
print("Overall:", overall_plot_path)
print("By split:", split_plot_path)

print("\nSplit counts:")
print(split_df["split"].value_counts())

print("\nGroup leakage check passed.")

In [ ]:
# Phase 2: Save report-ready split preview

preview = split_df[["image_id", "lesion_id", "dx", "split", "image_path"]].head(20)
preview_path = output_dirs["tables_dir"] / "split_preview.csv"
preview.to_csv(preview_path, index=False)

print("Saved preview:", preview_path)
display(preview)

In [ ]:
# Phase 3: Update repository from GitHub

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull"], check=True)

print("Updated repository.")
print("Current directory:", Path.cwd())

In [ ]:
# Debug: inspect current src.data functions

import src.data as data

print("Functions currently in src.data:")
for name in dir(data):
    if not name.startswith("_"):
        print(name)

In [ ]:
# Debug: search for expected Phase 3 function names in src/data.py

from pathlib import Path

data_file = Path("src/data.py")
text = data_file.read_text(encoding="utf-8")

expected = [
    "get_label_mapping",
    "add_label_indices",
    "compute_class_weights",
    "HAM10000ImageDataset",
    "get_image_transforms",
    "create_dataloader",
    "load_split_csvs",
    "build_datasets_and_loaders",
]

for name in expected:
    print(f"{name}:", name in text)

In [ ]:
# Phase 3: Reload updated source modules after git pull

import importlib
import src.data
import src.models
import src.plots

importlib.reload(src.data)
importlib.reload(src.models)
importlib.reload(src.plots)

print("Reloaded src.data, src.models, and src.plots")

print("\nUpdated functions in src.data:")
for name in [
    "get_label_mapping",
    "add_label_indices",
    "compute_class_weights",
    "HAM10000ImageDataset",
    "get_image_transforms",
    "create_dataloader",
    "load_split_csvs",
    "build_datasets_and_loaders",
]:
    print(name, ":", hasattr(src.data, name))

In [ ]:
# Phase 3: Import check

from src.config import load_config
from src.paths import ensure_output_dirs
from src.data import (
    load_split_csvs,
    build_datasets_and_loaders,
    compute_class_weights,
)
from src.models import (
    create_efficientnet_classifier,
    summarize_model,
)
from src.plots import plot_batch_preview

print("Phase 3 imports successful.")

In [ ]:
# Phase 3: Build PyTorch datasets and dataloaders

from pathlib import Path
import pandas as pd
import torch

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

split_dir = output_dirs["tables_dir"]

datasets, dataloaders, class_weights_df = build_datasets_and_loaders(
    split_dir=split_dir,
    classes=config["data"]["classes"],
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    fallback_batch_size=config["training"]["fallback_batch_size"],
    num_workers=2,
)

print("Datasets:")
for split_name, dataset in datasets.items():
    print(f"{split_name}: {len(dataset)} samples")

print("\nDataloaders:")
for split_name, loader in dataloaders.items():
    print(f"{split_name}: batch_size={loader.batch_size}, batches={len(loader)}")

print("\nClass weights:")
display(class_weights_df)

class_weights_path = output_dirs["tables_dir"] / "class_weights.csv"
class_weights_df.to_csv(class_weights_path, index=False)
print("Saved class weights:", class_weights_path)

In [ ]:
# Phase 3: One-batch DataLoader sanity check

batch = next(iter(dataloaders["train"]))

print("Batch keys:", batch.keys())
print("Image tensor shape:", batch["image"].shape)
print("Label tensor shape:", batch["label"].shape)
print("Image dtype:", batch["image"].dtype)
print("Label dtype:", batch["label"].dtype)

print("\nFirst 10 labels:", batch["label"][:10])
print("First 10 class names:", batch["class_name"][:10])
print("First 10 image IDs:", batch["image_id"][:10])

preview_path = plot_batch_preview(
    batch=batch,
    output_path=output_dirs["figures_dir"] / "phase3_train_batch_preview.png",
    max_images=8,
)

print("Saved batch preview:", preview_path)

In [ ]:
# Phase 3: EfficientNet-B0 model sanity check

import torch
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
    dropout_probability=config["model"]["dropout_probability"],
)

model = model.to(device)
model.eval()

images = batch["image"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():
    logits = model(images)

print("Input image shape:", images.shape)
print("Logits shape:", logits.shape)
print("Expected logits shape:", (images.shape[0], config["model"]["num_classes"]))

assert logits.shape == (images.shape[0], config["model"]["num_classes"])

model_summary_df = summarize_model(
    model=model,
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
)

model_summary_path = output_dirs["tables_dir"] / "phase3_model_summary.csv"
model_summary_df.to_csv(model_summary_path, index=False)

print("\nModel summary:")
display(model_summary_df)
print("Saved model summary:", model_summary_path)

In [ ]:
# Phase 3: Weighted loss sanity check

import torch

weights = torch.tensor(
    class_weights_df["weight"].values,
    dtype=torch.float32,
    device=device,
)

criterion = torch.nn.CrossEntropyLoss(weight=weights)

model.train()
logits = model(images)
loss = criterion(logits, labels)

print("Loss:", float(loss.detach().cpu()))
print("Loss is finite:", torch.isfinite(loss).item())

In [ ]:
from pathlib import Path

for path in [
    "outputs/tables/train_split.csv",
    "outputs/tables/val_split.csv",
    "outputs/tables/test_split.csv",
    "outputs/tables/class_weights.csv",
    "outputs/tables/phase3_model_summary.csv",
    "outputs/figures/phase3_train_batch_preview.png",
]:
    print(path, Path(path).exists())

In [ ]:
# Phase 4: Update repository from GitHub

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull"], check=True)

print("Updated repository.")
print("Current directory:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

In [ ]:
# Phase 4: Reload updated modules

import importlib
import src.train
import src.inference
import src.metrics
import src.plots

importlib.reload(src.train)
importlib.reload(src.inference)
importlib.reload(src.metrics)
importlib.reload(src.plots)

print("Phase 4 modules reloaded.")

In [ ]:
# Phase 4: Import check

from src.train import (
    create_weighted_cross_entropy_loss,
    create_optimizer,
    train_baseline_model,
    evaluate_model,
)
from src.inference import (
    load_checkpoint,
    collect_logits,
    save_logits_and_labels,
    prediction_dataframe_from_logits,
)
from src.metrics import (
    classification_metrics_from_logits,
    classification_report_dataframe,
    confusion_matrix_dataframe,
)
from src.plots import (
    plot_training_curves,
    plot_confusion_matrix,
)

print("Phase 4 imports successful.")

In [ ]:
# Phase 4: Prepare dataloaders, model, loss, optimizer

from pathlib import Path
import torch
import pandas as pd

from src.config import load_config
from src.paths import ensure_output_dirs
from src.reproducibility import set_global_seed
from src.data import build_datasets_and_loaders
from src.models import create_efficientnet_classifier, summarize_model
from src.train import create_weighted_cross_entropy_loss, create_optimizer

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

seed = 42
set_global_seed(seed, deterministic=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

datasets, dataloaders, class_weights_df = build_datasets_and_loaders(
    split_dir=output_dirs["tables_dir"],
    classes=config["data"]["classes"],
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    fallback_batch_size=config["training"]["fallback_batch_size"],
    num_workers=2,
)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
    dropout_probability=config["model"]["dropout_probability"],
).to(device)

criterion = create_weighted_cross_entropy_loss(
    class_weights_df=class_weights_df,
    device=device,
)

optimizer = create_optimizer(
    model=model,
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
)

print("Datasets:")
for split_name, dataset in datasets.items():
    print(split_name, len(dataset))

print("\nDataloaders:")
for split_name, loader in dataloaders.items():
    print(split_name, "batch_size:", loader.batch_size, "batches:", len(loader))

model_summary_df = summarize_model(
    model=model,
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
)

display(model_summary_df)

In [ ]:
# Phase 4: One-epoch timing test

import time
from src.train import train_one_epoch, evaluate_model

start_time = time.time()

train_metrics = train_one_epoch(
    model=model,
    dataloader=dataloaders["train"],
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

val_metrics, val_logits_tmp, val_labels_tmp = evaluate_model(
    model=model,
    dataloader=dataloaders["val"],
    criterion=criterion,
    device=device,
    classes=config["data"]["classes"],
    use_amp=config["training"]["mixed_precision"],
)

elapsed = time.time() - start_time

print("One epoch elapsed seconds:", round(elapsed, 2))
print("One epoch elapsed minutes:", round(elapsed / 60, 2))
print("Train metrics:", train_metrics)
print("Validation metrics:", val_metrics)
print("Validation logits shape:", val_logits_tmp.shape)
print("Validation labels shape:", val_labels_tmp.shape)

In [ ]:
# Phase 4: Reinitialize model for clean full training

from src.reproducibility import set_global_seed
from src.models import create_efficientnet_classifier
from src.train import create_weighted_cross_entropy_loss, create_optimizer, train_baseline_model

seed = 42
set_global_seed(seed, deterministic=False)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
    dropout_probability=config["model"]["dropout_probability"],
).to(device)

criterion = create_weighted_cross_entropy_loss(
    class_weights_df=class_weights_df,
    device=device,
)

optimizer = create_optimizer(
    model=model,
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
)

checkpoint_path = output_dirs["checkpoints_dir"] / f"model_seed{seed}.pth"
history_path = output_dirs["tables_dir"] / f"training_history_seed{seed}.csv"

print("Checkpoint path:", checkpoint_path)
print("History path:", history_path)
print("Max epochs:", config["training"]["max_epochs"])
print("Early stopping patience:", config["training"]["early_stopping_patience"])

In [ ]:
# Phase 4: Train baseline EfficientNet-B0 seed 42

history_df = train_baseline_model(
    model=model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    classes=config["data"]["classes"],
    max_epochs=config["training"]["max_epochs"],
    patience=config["training"]["early_stopping_patience"],
    checkpoint_path=checkpoint_path,
    history_path=history_path,
    seed=seed,
    use_amp=config["training"]["mixed_precision"],
)

display(history_df)
print("Training completed.")

In [ ]:
# Phase 4: Save training curves

from src.plots import plot_training_curves

training_curve_path = plot_training_curves(
    history_df=history_df,
    output_path=output_dirs["figures_dir"] / f"training_curves_seed{seed}.png",
)

print("Saved training curves:", training_curve_path)

In [ ]:
# Phase 4: Load best checkpoint and export validation/test logits

from src.inference import (
    load_checkpoint,
    collect_logits,
    save_logits_and_labels,
    prediction_dataframe_from_logits,
)

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

print("Loaded checkpoint epoch:", checkpoint.get("epoch"))
print("Best val macro F1:", checkpoint.get("best_val_macro_f1"))

val_logits, val_labels, val_metadata = collect_logits(
    model=model,
    dataloader=dataloaders["val"],
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

test_logits, test_labels, test_metadata = collect_logits(
    model=model,
    dataloader=dataloaders["test"],
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

val_saved = save_logits_and_labels(
    logits=val_logits,
    labels=val_labels,
    output_dir=output_dirs["logits_dir"],
    split_name="val",
    seed=seed,
)

test_saved = save_logits_and_labels(
    logits=test_logits,
    labels=test_labels,
    output_dir=output_dirs["logits_dir"],
    split_name="test",
    seed=seed,
)

print("Validation logits shape:", val_logits.shape)
print("Validation labels shape:", val_labels.shape)
print("Test logits shape:", test_logits.shape)
print("Test labels shape:", test_labels.shape)

print("Saved validation:", val_saved)
print("Saved test:", test_saved)

In [ ]:
# Phase 4: Compute baseline test metrics and save predictions

from src.metrics import (
    classification_metrics_from_logits,
    classification_report_dataframe,
    confusion_matrix_dataframe,
)
from src.inference import prediction_dataframe_from_logits
from src.plots import plot_confusion_matrix
import pandas as pd

classes = config["data"]["classes"]

test_metrics = classification_metrics_from_logits(
    logits=test_logits,
    labels=test_labels,
    classes=classes,
)

test_predictions_df = prediction_dataframe_from_logits(
    logits=test_logits,
    labels=test_labels,
    metadata_df=test_metadata,
    classes=classes,
)

test_predictions_path = output_dirs["predictions_dir"] / f"test_predictions_seed{seed}.csv"
test_predictions_df.to_csv(test_predictions_path, index=False)

test_pred_labels = test_predictions_df["predicted_label"].to_numpy()

classification_report_df = classification_report_dataframe(
    labels=test_labels,
    predictions=test_pred_labels,
    classes=classes,
)

confusion_df = confusion_matrix_dataframe(
    labels=test_labels,
    predictions=test_pred_labels,
    classes=classes,
)

metrics_df = pd.DataFrame([test_metrics])
metrics_path = output_dirs["tables_dir"] / f"baseline_seed{seed}_metrics.csv"
report_path = output_dirs["tables_dir"] / f"baseline_seed{seed}_classification_report.csv"
confusion_path = output_dirs["figures_dir"] / f"confusion_matrix_seed{seed}.png"

metrics_df.to_csv(metrics_path, index=False)
classification_report_df.to_csv(report_path)

plot_confusion_matrix(
    confusion_df=confusion_df,
    output_path=confusion_path,
)

print("Test metrics:")
display(metrics_df)

print("\nClassification report:")
display(classification_report_df)

print("\nConfusion matrix:")
display(confusion_df)

print("\nSaved:")
print("Metrics:", metrics_path)
print("Report:", report_path)
print("Predictions:", test_predictions_path)
print("Confusion matrix:", confusion_path)

In [ ]:
# Phase 4: Verify generated files

expected_files = [
    output_dirs["checkpoints_dir"] / f"model_seed{seed}.pth",
    output_dirs["tables_dir"] / f"training_history_seed{seed}.csv",
    output_dirs["tables_dir"] / f"baseline_seed{seed}_metrics.csv",
    output_dirs["tables_dir"] / f"baseline_seed{seed}_classification_report.csv",
    output_dirs["logits_dir"] / f"val_logits_seed{seed}.npy",
    output_dirs["logits_dir"] / f"val_labels_seed{seed}.npy",
    output_dirs["logits_dir"] / f"test_logits_seed{seed}.npy",
    output_dirs["logits_dir"] / f"test_labels_seed{seed}.npy",
    output_dirs["predictions_dir"] / f"test_predictions_seed{seed}.csv",
    output_dirs["figures_dir"] / f"training_curves_seed{seed}.png",
    output_dirs["figures_dir"] / f"confusion_matrix_seed{seed}.png",
]

for path in expected_files:
    print(path, path.exists())

In [ ]:
# Phase 5: Update repository from GitHub

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull"], check=True)

print("Updated repository.")
print("Current directory:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

In [ ]:
# Phase 5: Reload updated modules

import importlib
import src.calibration
import src.metrics
import src.inference
import src.plots

importlib.reload(src.calibration)
importlib.reload(src.metrics)
importlib.reload(src.inference)
importlib.reload(src.plots)

print("Phase 5 modules reloaded.")

In [ ]:
# Phase 5: Import check

from src.calibration import (
    fit_temperature_scaling,
    apply_temperature_scaling,
    save_temperature,
)
from src.metrics import (
    softmax_numpy,
    calibration_metrics_from_logits,
    reliability_curve_dataframe,
)
from src.inference import prediction_dataframe_from_probabilities
from src.plots import plot_reliability_diagram

print("Phase 5 imports successful.")

In [ ]:
# Phase 5: Load baseline logits and fit Temperature Scaling

import numpy as np
import pandas as pd
from pathlib import Path

from src.config import load_config
from src.paths import ensure_output_dirs
from src.calibration import fit_temperature_scaling, apply_temperature_scaling, save_temperature
from src.metrics import (
    softmax_numpy,
    calibration_metrics_from_logits,
    reliability_curve_dataframe,
)
from src.inference import prediction_dataframe_from_probabilities
from src.plots import plot_reliability_diagram

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

seed = 42
classes = config["data"]["classes"]

val_logits = np.load(output_dirs["logits_dir"] / f"val_logits_seed{seed}.npy")
val_labels = np.load(output_dirs["logits_dir"] / f"val_labels_seed{seed}.npy")
test_logits = np.load(output_dirs["logits_dir"] / f"test_logits_seed{seed}.npy")
test_labels = np.load(output_dirs["logits_dir"] / f"test_labels_seed{seed}.npy")

print("Val logits:", val_logits.shape)
print("Val labels:", val_labels.shape)
print("Test logits:", test_logits.shape)
print("Test labels:", test_labels.shape)

temperature = fit_temperature_scaling(
    val_logits=val_logits,
    val_labels=val_labels,
    max_iter=1000,
    lr=0.01,
    initial_temperature=1.0,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print("Learned temperature:", temperature)

temperature_path = output_dirs["tables_dir"] / f"temperature_seed{seed}.csv"
save_temperature(temperature, temperature_path)
print("Saved temperature:", temperature_path)

In [ ]:
# Phase 5: Compare uncalibrated vs calibrated test metrics

uncalibrated_metrics = calibration_metrics_from_logits(
    logits=test_logits,
    labels=test_labels,
    classes=classes,
    temperature=None,
    n_bins=15,
)

calibrated_metrics = calibration_metrics_from_logits(
    logits=test_logits,
    labels=test_labels,
    classes=classes,
    temperature=temperature,
    n_bins=15,
)

calibration_metrics_df = pd.DataFrame([
    {"model": "baseline_uncalibrated", **uncalibrated_metrics},
    {"model": "baseline_temperature_scaled", "temperature": temperature, **calibrated_metrics},
])

calibration_metrics_path = output_dirs["tables_dir"] / f"calibration_metrics_seed{seed}.csv"
calibration_metrics_df.to_csv(calibration_metrics_path, index=False)

print("Calibration metrics:")
display(calibration_metrics_df)

print("Saved calibration metrics:", calibration_metrics_path)

In [ ]:
# Phase 5: Save calibrated probabilities and calibrated predictions

uncalibrated_probs = softmax_numpy(test_logits)
calibrated_logits = apply_temperature_scaling(test_logits, temperature)
calibrated_probs = softmax_numpy(calibrated_logits)

uncalibrated_probs_path = output_dirs["logits_dir"] / f"test_probs_uncalibrated_seed{seed}.npy"
calibrated_probs_path = output_dirs["logits_dir"] / f"test_probs_calibrated_seed{seed}.npy"

np.save(uncalibrated_probs_path, uncalibrated_probs)
np.save(calibrated_probs_path, calibrated_probs)

print("Saved uncalibrated probs:", uncalibrated_probs_path)
print("Saved calibrated probs:", calibrated_probs_path)

# Load existing metadata from Phase 4 predictions
baseline_predictions_path = output_dirs["predictions_dir"] / f"test_predictions_seed{seed}.csv"
baseline_predictions_df = pd.read_csv(baseline_predictions_path)

metadata_df = baseline_predictions_df[["image_id", "lesion_id", "true_class"]].copy()
metadata_df["class_name"] = metadata_df["true_class"]

calibrated_predictions_df = prediction_dataframe_from_probabilities(
    probs=calibrated_probs,
    labels=test_labels,
    metadata_df=metadata_df,
    classes=classes,
)

calibrated_predictions_path = output_dirs["predictions_dir"] / f"test_predictions_calibrated_seed{seed}.csv"
calibrated_predictions_df.to_csv(calibrated_predictions_path, index=False)

print("Saved calibrated predictions:", calibrated_predictions_path)
display(calibrated_predictions_df.head())

In [ ]:
# Phase 5: Reliability diagram before and after Temperature Scaling

uncalibrated_curve_df = reliability_curve_dataframe(
    probs=uncalibrated_probs,
    labels=test_labels,
    n_bins=15,
)

calibrated_curve_df = reliability_curve_dataframe(
    probs=calibrated_probs,
    labels=test_labels,
    n_bins=15,
)

uncalibrated_curve_path = output_dirs["tables_dir"] / f"reliability_curve_uncalibrated_seed{seed}.csv"
calibrated_curve_path = output_dirs["tables_dir"] / f"reliability_curve_calibrated_seed{seed}.csv"

uncalibrated_curve_df.to_csv(uncalibrated_curve_path, index=False)
calibrated_curve_df.to_csv(calibrated_curve_path, index=False)

reliability_plot_path = plot_reliability_diagram(
    uncalibrated_curve_df=uncalibrated_curve_df,
    calibrated_curve_df=calibrated_curve_df,
    output_path=output_dirs["figures_dir"] / f"reliability_diagram_seed{seed}.png",
)

print("Saved reliability curve tables:")
print(uncalibrated_curve_path)
print(calibrated_curve_path)
print("Saved reliability diagram:", reliability_plot_path)

In [ ]:
# Phase 5: Verify generated files

expected_files = [
    output_dirs["tables_dir"] / f"temperature_seed{seed}.csv",
    output_dirs["tables_dir"] / f"calibration_metrics_seed{seed}.csv",
    output_dirs["tables_dir"] / f"reliability_curve_uncalibrated_seed{seed}.csv",
    output_dirs["tables_dir"] / f"reliability_curve_calibrated_seed{seed}.csv",
    output_dirs["logits_dir"] / f"test_probs_uncalibrated_seed{seed}.npy",
    output_dirs["logits_dir"] / f"test_probs_calibrated_seed{seed}.npy",
    output_dirs["predictions_dir"] / f"test_predictions_calibrated_seed{seed}.csv",
    output_dirs["figures_dir"] / f"reliability_diagram_seed{seed}.png",
]

for path in expected_files:
    print(path, path.exists())

### 2nd Training - Resume Session for Phase 6

restored the persisted outputs and loads the latest source code without rerunning the completed Phase 1–5 experiments.

In [1]:
# Phase 6:

import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("/kaggle/working exists:", Path("/kaggle/working").exists())
print("/kaggle/input exists:", Path("/kaggle/input").exists())

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch check failed:", repr(exc))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
/kaggle/working exists: True
/kaggle/input exists: True
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# Phase 6:

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/DiloJan-26/skin-lesion-uncertainty.git"
REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")

if (REPO_DIR / ".git").exists():
    os.chdir(REPO_DIR)

    tracked_status = subprocess.run(
        ["git", "status", "--short", "--untracked-files=no"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()

    if tracked_status:
        print("Tracked local changes detected:")
        print(tracked_status)
        raise RuntimeError("Stop here and review tracked local changes before pulling.")

    subprocess.run(["git", "pull", "--ff-only"], check=True)
    print("Existing repository updated.")

elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository."
    )

else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print("Repository cloned.")

os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-4"], check=True)

Updating f4a19f2..36b41d3
Fast-forward
 README.md                     |  10 ++-
 src/plots.py                  |  71 ++++++++++++++++++
 src/referral.py               | 167 ++++++++++++++++++++++++++++++++++++++++--
 src/uncertainty.py            | 103 ++++++++++++++++++++++++--
 tests/test_phase6_referral.py | 166 +++++++++++++++++++++++++++++++++++++++++
 5 files changed, 503 insertions(+), 14 deletions(-)
 create mode 100644 tests/test_phase6_referral.py
Existing repository updated.
Current directory: /kaggle/working/skin-lesion-uncertainty
36b41d3 Phase 6 uncertainty scoring and referral evaluation
f4a19f2 Phase 5 temperature scaling calibration
fccec8e Phase 4 baseline training utilities
b4c2685 Phase 3 dataset loaders and EfficientNet model


From https://github.com/DiloJan-26/skin-lesion-uncertainty
   f4a19f2..36b41d3  main       -> origin/main


CompletedProcess(args=['git', 'log', '--oneline', '-4'], returncode=0)

In [3]:
%pip install -q "timm>=0.9" "torchmetrics>=1.3" "PyYAML>=6.0"

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Phase 6:

from pathlib import Path

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
OUTPUTS = REPO_DIR / "outputs"

required_files = [
    OUTPUTS / "tables/train_split.csv",
    OUTPUTS / "tables/val_split.csv",
    OUTPUTS / "tables/test_split.csv",

    OUTPUTS / "checkpoints/model_seed42.pth",
    OUTPUTS / "tables/training_history_seed42.csv",
    OUTPUTS / "tables/baseline_seed42_metrics.csv",

    OUTPUTS / "logits/val_logits_seed42.npy",
    OUTPUTS / "logits/val_labels_seed42.npy",
    OUTPUTS / "logits/test_logits_seed42.npy",
    OUTPUTS / "logits/test_labels_seed42.npy",

    OUTPUTS / "tables/temperature_seed42.csv",
    OUTPUTS / "tables/calibration_metrics_seed42.csv",
    OUTPUTS / "logits/test_probs_uncalibrated_seed42.npy",
    OUTPUTS / "logits/test_probs_calibrated_seed42.npy",
    OUTPUTS / "predictions/test_predictions_calibrated_seed42.csv",
    OUTPUTS / "figures/reliability_diagram_seed42.png",
]

missing_files = []

for path in required_files:
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    print(f"{exists!s:5} | {size:>10} bytes | {path.relative_to(REPO_DIR)}")

    if not exists or size == 0:
        missing_files.append(path)

if missing_files:
    print("\nMissing or empty files:")
    for path in missing_files:
        print(" -", path)
else:
    print("\nAll required Phase 2–5 artifacts are available.")

True  |    1276011 bytes | outputs/tables/train_split.csv
True  |     159032 bytes | outputs/tables/val_split.csv
True  |     156920 bytes | outputs/tables/test_split.csv
True  |   48676754 bytes | outputs/checkpoints/model_seed42.pth
True  |       2209 bytes | outputs/tables/training_history_seed42.csv
True  |        145 bytes | outputs/tables/baseline_seed42_metrics.csv
True  |      14268 bytes | outputs/logits/val_logits_seed42.npy
True  |       8208 bytes | outputs/logits/val_labels_seed42.npy
True  |      14016 bytes | outputs/logits/test_logits_seed42.npy
True  |       8064 bytes | outputs/logits/test_labels_seed42.npy
True  |         39 bytes | outputs/tables/temperature_seed42.csv
True  |        244 bytes | outputs/tables/calibration_metrics_seed42.csv
True  |      55680 bytes | outputs/logits/test_probs_uncalibrated_seed42.npy
True  |      55680 bytes | outputs/logits/test_probs_calibrated_seed42.npy
True  |     206325 bytes | outputs/predictions/test_predictions_calibrated_se

In [5]:
# Phase 6:

from pathlib import Path
from datetime import datetime
import shutil

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_base = Path("/kaggle/working") / f"phase5_outputs_backup_{timestamp}"

backup_zip = shutil.make_archive(
    base_name=str(backup_base),
    format="zip",
    root_dir=REPO_DIR,
    base_dir="outputs",
)

print("Backup created:", backup_zip)
print("Backup size:", Path(backup_zip).stat().st_size, "bytes")

Backup created: /kaggle/working/phase5_outputs_backup_20260725_104254.zip
Backup size: 46617282 bytes


In [6]:
# Phase 6:

from pathlib import Path
import json
import numpy as np
import pandas as pd

from src.config import load_config
from src.paths import ensure_output_dirs
from src.calibration import apply_temperature_scaling
from src.metrics import softmax_numpy

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

seed = 42
classes = config["data"]["classes"]
coverages = config["referral"]["coverage_levels"]

val_logits = np.load(
    output_dirs["logits_dir"] / f"val_logits_seed{seed}.npy"
)
val_labels = np.load(
    output_dirs["logits_dir"] / f"val_labels_seed{seed}.npy"
)
test_logits = np.load(
    output_dirs["logits_dir"] / f"test_logits_seed{seed}.npy"
)
test_labels = np.load(
    output_dirs["logits_dir"] / f"test_labels_seed{seed}.npy"
)
test_probs_calibrated = np.load(
    output_dirs["logits_dir"] / f"test_probs_calibrated_seed{seed}.npy"
)

temperature_path = (
    output_dirs["tables_dir"] / f"temperature_seed{seed}.csv"
)

with temperature_path.open("r", encoding="utf-8") as file:
    temperature_payload = json.load(file)

if isinstance(temperature_payload, dict):
    temperature = float(temperature_payload["temperature"])
else:
    temperature = float(temperature_payload)

val_probs_calibrated = softmax_numpy(
    apply_temperature_scaling(val_logits, temperature)
)

baseline_predictions_df = pd.read_csv(
    output_dirs["predictions_dir"]
    / f"test_predictions_calibrated_seed{seed}.csv"
)

print("Temperature:", temperature)
print("Validation logits:", val_logits.shape)
print("Validation calibrated probabilities:", val_probs_calibrated.shape)
print("Test calibrated probabilities:", test_probs_calibrated.shape)
print("Test labels:", test_labels.shape)
print("Prediction rows:", len(baseline_predictions_df))

Temperature: 1.740342140197754
Validation logits: (1010, 7)
Validation calibrated probabilities: (1010, 7)
Test calibrated probabilities: (992, 7)
Test labels: (992,)
Prediction rows: 992


In [7]:
# Phase 6:

import importlib
import src.uncertainty
import src.referral
import src.plots

importlib.reload(src.uncertainty)
importlib.reload(src.referral)
importlib.reload(src.plots)

from src.uncertainty import (
    confidence_uncertainty,
    predictive_entropy,
    uncertainty_scores_dataframe,
    uncertainty_error_detection_table,
)

from src.referral import (
    referral_results_at_coverages,
    select_uncertainty_threshold_for_coverage,
    evaluate_referral_threshold,
    add_referral_decision,
)

from src.plots import (
    plot_uncertainty_correct_vs_incorrect,
    plot_risk_coverage_curve,
)

print("Phase 6 imports successful.")

Phase 6 imports successful.


In [8]:
# Phase 6:

metadata_df = baseline_predictions_df[["image_id", "lesion_id"]].copy()

scores_df = uncertainty_scores_dataframe(
    probs=test_probs_calibrated,
    labels=test_labels,
    classes=classes,
    metadata_df=metadata_df,
)

scores_path = (
    output_dirs["tables_dir"]
    / f"baseline_uncertainty_scores_seed{seed}.csv"
)

scores_df.to_csv(scores_path, index=False)

print("Saved uncertainty scores:", scores_path)
display(scores_df.head())

print("\nUncertainty summary:")
display(
    scores_df[
        [
            "confidence",
            "confidence_uncertainty",
            "predictive_entropy",
            "correct",
        ]
    ].describe()
)

Saved uncertainty scores: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_uncertainty_scores_seed42.csv


,image_id,lesion_id,true_label,true_class,predicted_label,predicted_class,confidence,confidence_uncertainty,predictive_entropy,correct
0,ISIC_0027419,HAM_0000118,2,bkl,2,bkl,0.816670,0.183330,0.736413,True
1,ISIC_0025030,HAM_0000118,2,bkl,2,bkl,0.888796,0.111204,0.464610,True
2,ISIC_0029176,HAM_0002761,2,bkl,2,bkl,0.803748,0.196252,0.721897,True
3,ISIC_0029068,HAM_0002761,2,bkl,1,bcc,0.395096,0.604904,1.512238,False
4,ISIC_0032343,HAM_0006071,2,bkl,2,bkl,0.717796,0.282204,0.745596,True



Uncertainty summary:


,confidence,confidence_uncertainty,predictive_entropy
count,992.000000,992.000000,9.920000e+02
mean,0.818613,0.181387,4.831694e-01
std,0.203826,0.203826,4.710938e-01
min,0.238987,0.000000,1.657861e-10
25%,0.652236,0.003394,2.471448e-02
50%,0.920093,0.079907,3.399606e-01
75%,0.996606,0.347764,8.623219e-01
max,1.000000,0.761013,1.834149e+00


In [9]:
# Phase 6:

error_detection_df = uncertainty_error_detection_table(
    scores_df=scores_df,
    score_columns=[
        "confidence_uncertainty",
        "predictive_entropy",
    ],
)

error_detection_path = (
    output_dirs["tables_dir"]
    / f"baseline_uncertainty_error_detection_seed{seed}.csv"
)

error_detection_df.to_csv(error_detection_path, index=False)

print("Uncertainty error-detection AUROC:")
display(error_detection_df)

print("Saved:", error_detection_path)

Uncertainty error-detection AUROC:


,score,error_detection_auroc
0,confidence_uncertainty,0.843412
1,predictive_entropy,0.838049


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_uncertainty_error_detection_seed42.csv


In [10]:
# Phase 6:

confidence_referral_df = referral_results_at_coverages(
    probs=test_probs_calibrated,
    labels=test_labels,
    uncertainty_scores=scores_df["confidence_uncertainty"].to_numpy(),
    coverages=coverages,
    classes=classes,
    method_name="calibrated_confidence_uncertainty",
)

entropy_referral_df = referral_results_at_coverages(
    probs=test_probs_calibrated,
    labels=test_labels,
    uncertainty_scores=scores_df["predictive_entropy"].to_numpy(),
    coverages=coverages,
    classes=classes,
    method_name="calibrated_predictive_entropy",
)

referral_results_df = pd.concat(
    [confidence_referral_df, entropy_referral_df],
    ignore_index=True,
)

referral_results_path = (
    output_dirs["tables_dir"]
    / f"baseline_referral_results_seed{seed}.csv"
)

referral_results_df.to_csv(referral_results_path, index=False)

print("Referral results:")
display(referral_results_df)

print("Saved:", referral_results_path)

Referral results:


,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1
0,calibrated_confidence_uncertainty,1.000000,0.000000,992,0,0.812500,0.187500,0.714404
1,calibrated_confidence_uncertainty,0.950605,0.049395,943,49,0.834571,0.165429,0.736751
2,calibrated_confidence_uncertainty,0.900202,0.099798,893,99,0.849944,0.150056,0.757617
3,calibrated_confidence_uncertainty,0.850806,0.149194,844,148,0.864929,0.135071,0.777277
4,calibrated_confidence_uncertainty,0.800403,0.199597,794,198,0.886650,0.113350,0.807588
5,calibrated_confidence_uncertainty,0.700605,0.299395,695,297,0.923741,0.076259,0.850408
6,calibrated_confidence_uncertainty,0.600806,0.399194,596,396,0.953020,0.046980,0.878084
7,calibrated_predictive_entropy,1.000000,0.000000,992,0,0.812500,0.187500,0.714404
8,calibrated_predictive_entropy,0.950605,0.049395,943,49,0.834571,0.165429,0.750202
9,calibrated_predictive_entropy,0.900202,0.099798,893,99,0.848824,0.151176,0.770546


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_referral_results_seed42.csv


In [11]:
# Phase 6:

val_conf_uncertainty = confidence_uncertainty(val_probs_calibrated)

target_coverage = 0.90

selected_threshold = select_uncertainty_threshold_for_coverage(
    uncertainty_scores=val_conf_uncertainty,
    target_coverage=target_coverage,
)

selected_result = evaluate_referral_threshold(
    probs=test_probs_calibrated,
    labels=test_labels,
    uncertainty_scores=scores_df["confidence_uncertainty"].to_numpy(),
    threshold=selected_threshold,
    classes=classes,
    method_name="validation_selected_confidence_uncertainty",
)

threshold_df = pd.DataFrame([{
    "method": "confidence_uncertainty",
    "target_validation_coverage": target_coverage,
    "selected_threshold": selected_threshold,
}])

selected_result_df = pd.DataFrame([selected_result])

threshold_path = (
    output_dirs["tables_dir"]
    / f"baseline_selected_referral_threshold_seed{seed}.csv"
)

selected_result_path = (
    output_dirs["tables_dir"]
    / f"baseline_selected_referral_test_result_seed{seed}.csv"
)

threshold_df.to_csv(threshold_path, index=False)
selected_result_df.to_csv(selected_result_path, index=False)

scores_with_decision_df = add_referral_decision(
    scores_df=scores_df,
    uncertainty_column="confidence_uncertainty",
    threshold=selected_threshold,
)

decision_path = (
    output_dirs["predictions_dir"]
    / f"test_referral_decisions_seed{seed}.csv"
)

scores_with_decision_df.to_csv(decision_path, index=False)

print("Selected threshold:")
display(threshold_df)

print("\nTest result using validation-selected threshold:")
display(selected_result_df)

print("\nDecision counts:")
display(
    scores_with_decision_df["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="count")
)

Selected threshold:


,method,target_validation_coverage,selected_threshold
0,confidence_uncertainty,0.9,0.499722



Test result using validation-selected threshold:


,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1,threshold
0,validation_selected_confidence_uncertainty,0.893145,0.106855,886,106,0.85553,0.14447,0.76686,0.499722



Decision counts:


,decision,count
0,accept,886
1,refer,106


In [12]:
# Phase 6:

uncertainty_plot_path = plot_uncertainty_correct_vs_incorrect(
    scores_df=scores_df,
    score_column="confidence_uncertainty",
    output_path=(
        output_dirs["figures_dir"]
        / f"baseline_uncertainty_correct_vs_incorrect_seed{seed}.png"
    ),
)

risk_coverage_plot_path = plot_risk_coverage_curve(
    referral_results_df=referral_results_df,
    output_path=(
        output_dirs["figures_dir"]
        / f"baseline_risk_coverage_curve_seed{seed}.png"
    ),
)

print("Saved uncertainty plot:", uncertainty_plot_path)
print("Saved risk-coverage curve:", risk_coverage_plot_path)

Saved uncertainty plot: /kaggle/working/skin-lesion-uncertainty/outputs/figures/baseline_uncertainty_correct_vs_incorrect_seed42.png
Saved risk-coverage curve: /kaggle/working/skin-lesion-uncertainty/outputs/figures/baseline_risk_coverage_curve_seed42.png


In [13]:
# Phase 6:

expected_files = [
    output_dirs["tables_dir"]
    / f"baseline_uncertainty_scores_seed{seed}.csv",

    output_dirs["tables_dir"]
    / f"baseline_uncertainty_error_detection_seed{seed}.csv",

    output_dirs["tables_dir"]
    / f"baseline_referral_results_seed{seed}.csv",

    output_dirs["tables_dir"]
    / f"baseline_selected_referral_threshold_seed{seed}.csv",

    output_dirs["tables_dir"]
    / f"baseline_selected_referral_test_result_seed{seed}.csv",

    output_dirs["predictions_dir"]
    / f"test_referral_decisions_seed{seed}.csv",

    output_dirs["figures_dir"]
    / f"baseline_uncertainty_correct_vs_incorrect_seed{seed}.png",

    output_dirs["figures_dir"]
    / f"baseline_risk_coverage_curve_seed{seed}.png",
]

for path in expected_files:
    print(path, path.exists())

/kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_uncertainty_scores_seed42.csv True
/kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_uncertainty_error_detection_seed42.csv True
/kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_referral_results_seed42.csv True
/kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_selected_referral_threshold_seed42.csv True
/kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_selected_referral_test_result_seed42.csv True
/kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_referral_decisions_seed42.csv True
/kaggle/working/skin-lesion-uncertainty/outputs/figures/baseline_uncertainty_correct_vs_incorrect_seed42.png True
/kaggle/working/skin-lesion-uncertainty/outputs/figures/baseline_risk_coverage_curve_seed42.png True


### Phase 7 — Train Remaining Deep Ensemble Members

In [14]:
# Phase 7: current kaggle state

from pathlib import Path
import os
import torch

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

print("Repository:", Path.cwd())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

required_seed42_files = [
    REPO_DIR / "outputs/checkpoints/model_seed42.pth",
    REPO_DIR / "outputs/logits/val_logits_seed42.npy",
    REPO_DIR / "outputs/logits/test_logits_seed42.npy",
]

for path in required_seed42_files:
    print(path.name, path.exists())

Repository: /kaggle/working/skin-lesion-uncertainty
CUDA available: True
GPU: Tesla T4
model_seed42.pth True
val_logits_seed42.npy True
test_logits_seed42.npy True


In [15]:
# Phase 7: Reconstruct dataloaders

import pandas as pd
import torch

from src.config import load_config
from src.paths import ensure_output_dirs
from src.data import build_datasets_and_loaders

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classes = config["data"]["classes"]

datasets, dataloaders, class_weights_df = build_datasets_and_loaders(
    split_dir=output_dirs["tables_dir"],
    classes=classes,
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    fallback_batch_size=config["training"]["fallback_batch_size"],
    num_workers=2,
)

print("Device:", device)

for split_name, dataset in datasets.items():
    print(split_name, len(dataset))

for split_name, loader in dataloaders.items():
    print(
        split_name,
        "batch size:",
        loader.batch_size,
        "batches:",
        len(loader),
    )

display(class_weights_df)

Device: cuda
train 8013
val 1010
test 992
train batch size: 32 batches: 251
val batch size: 32 batches: 32
test batch size: 32 batches: 31


,class,count,weight
0,akiec,244,4.691452
1,bcc,398,2.876167
2,bkl,881,1.299335
3,df,94,12.177812
4,mel,881,1.299335
5,nv,5397,0.212102
6,vasc,118,9.700969


In [17]:
# Phase 7: Import existing training utilities

from src.reproducibility import set_global_seed
from src.models import create_efficientnet_classifier
from src.train import (
    create_weighted_cross_entropy_loss,
    create_optimizer,
    train_baseline_model,
)
from src.inference import (
    load_checkpoint,
    collect_logits,
    save_logits_and_labels,
    prediction_dataframe_from_logits,
)
from src.metrics import (
    classification_metrics_from_logits,
    classification_report_dataframe,
    confusion_matrix_dataframe,
)
from src.plots import (
    plot_training_curves,
    plot_confusion_matrix,
)

print("imports successful.")

imports successful.


In [18]:
# Phase 7: Train seed 123

import pandas as pd
import torch

seed = 123
set_global_seed(seed, deterministic=False)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
    dropout_probability=config["model"]["dropout_probability"],
).to(device)

criterion = create_weighted_cross_entropy_loss(
    class_weights_df=class_weights_df,
    device=device,
)

optimizer = create_optimizer(
    model=model,
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
)

checkpoint_path = (
    output_dirs["checkpoints_dir"]
    / f"model_seed{seed}.pth"
)

history_path = (
    output_dirs["tables_dir"]
    / f"training_history_seed{seed}.csv"
)

print("Training seed:", seed)
print("Checkpoint:", checkpoint_path)
print("History:", history_path)
print("Maximum epochs:", config["training"]["max_epochs"])
print(
    "Early stopping patience:",
    config["training"]["early_stopping_patience"],
)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Training seed: 123
Checkpoint: /kaggle/working/skin-lesion-uncertainty/outputs/checkpoints/model_seed123.pth
History: /kaggle/working/skin-lesion-uncertainty/outputs/tables/training_history_seed123.csv
Maximum epochs: 12
Early stopping patience: 3


In [19]:


history_df_123 = train_baseline_model(
    model=model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    classes=classes,
    max_epochs=config["training"]["max_epochs"],
    patience=config["training"]["early_stopping_patience"],
    checkpoint_path=checkpoint_path,
    history_path=history_path,
    seed=seed,
    use_amp=config["training"]["mixed_precision"],
)

display(history_df_123)

print("Seed 123 training completed.")

/kaggle/working/skin-lesion-uncertainty/src/train.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


train:   0%|          | 0/251 [00:00<?, ?it/s]

/kaggle/working/skin-lesion-uncertainty/src/train.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
/kaggle/working/skin-lesion-uncertainty/src/train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


Epoch 1/12 - train_loss=1.6732 train_acc=0.5315 val_loss=0.9318 val_macro_f1=0.5420


/kaggle/working/skin-lesion-uncertainty/src/train.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


...

train:   0%|          | 0/251 [00:00<?, ?it/s]

/kaggle/working/skin-lesion-uncertainty/src/train.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
/kaggle/working/skin-lesion-uncertainty/src/train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


Epoch 6/12 - train_loss=0.4173 train_acc=0.7973 val_loss=1.0683 val_macro_f1=0.6130
Early stopping after 6 epochs.


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,val_melanoma_recall,best_val_macro_f1
0,1,1.673160,0.531511,0.931843,0.637624,0.490733,0.707354,0.541987,0.371901,0.541987
1,2,0.884213,0.674279,0.749934,0.708911,0.530770,0.778233,0.604748,0.413223,0.604748
2,3,0.694948,0.732060,0.818655,0.715842,0.657786,0.765279,0.681543,0.776860,0.681543
3,4,0.619455,0.745039,0.847348,0.747525,0.538925,0.758472,0.596019,0.669421,0.681543
4,5,0.548482,0.784101,0.866198,0.701980,0.628756,0.769822,0.655917,0.809917,0.681543
5,6,0.417255,0.797329,1.068330,0.693069,0.594144,0.787277,0.612970,0.702479,0.681543


Seed 123 training completed.


In [26]:
# Phase 7: Save training curves

seed = 123

training_curve_path_123 = plot_training_curves(
    history_df=history_df_123,
    output_path=(
        output_dirs["figures_dir"]
        / f"training_curves_seed{seed}.png"
    ),
)

print("Saved:", training_curve_path_123)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/training_curves_seed123.png


In [27]:
# Phase 7: Load the best checkpoint and collect logits

checkpoint_123 = load_checkpoint(
    model=model,
    checkpoint_path=(
        output_dirs["checkpoints_dir"]
        / f"model_seed{seed}.pth"
    ),
    device=device,
)

print("Best checkpoint epoch:", checkpoint_123.get("epoch"))
print(
    "Best validation macro F1:",
    checkpoint_123.get("best_val_macro_f1"),
)

val_logits_123, val_labels_123, val_metadata_123 = collect_logits(
    model=model,
    dataloader=dataloaders["val"],
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

test_logits_123, test_labels_123, test_metadata_123 = collect_logits(
    model=model,
    dataloader=dataloaders["test"],
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

save_logits_and_labels(
    logits=val_logits_123,
    labels=val_labels_123,
    output_dir=output_dirs["logits_dir"],
    split_name="val",
    seed=seed,
)

save_logits_and_labels(
    logits=test_logits_123,
    labels=test_labels_123,
    output_dir=output_dirs["logits_dir"],
    split_name="test",
    seed=seed,
)

print("Validation logits:", val_logits_123.shape)
print("Validation labels:", val_labels_123.shape)
print("Test logits:", test_logits_123.shape)
print("Test labels:", test_labels_123.shape)

Best checkpoint epoch: 3
Best validation macro F1: 0.6815430826225234


/kaggle/working/skin-lesion-uncertainty/src/inference.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


Validation logits: (1010, 7)
Validation labels: (1010,)
Test logits: (992, 7)
Test labels: (992,)


In [28]:
# Phase 7: Compute seed 123 metrics

test_metrics_123 = classification_metrics_from_logits(
    logits=test_logits_123,
    labels=test_labels_123,
    classes=classes,
)

predictions_df_123 = prediction_dataframe_from_logits(
    logits=test_logits_123,
    labels=test_labels_123,
    metadata_df=test_metadata_123,
    classes=classes,
)

predictions_path_123 = (
    output_dirs["predictions_dir"]
    / f"test_predictions_seed{seed}.csv"
)

predictions_df_123.to_csv(
    predictions_path_123,
    index=False,
)

predicted_labels_123 = (
    predictions_df_123["predicted_label"].to_numpy()
)

report_df_123 = classification_report_dataframe(
    labels=test_labels_123,
    predictions=predicted_labels_123,
    classes=classes,
)

confusion_df_123 = confusion_matrix_dataframe(
    labels=test_labels_123,
    predictions=predicted_labels_123,
    classes=classes,
)

metrics_df_123 = pd.DataFrame([test_metrics_123])

metrics_path_123 = (
    output_dirs["tables_dir"]
    / f"baseline_seed{seed}_metrics.csv"
)

report_path_123 = (
    output_dirs["tables_dir"]
    / f"baseline_seed{seed}_classification_report.csv"
)

confusion_path_123 = (
    output_dirs["figures_dir"]
    / f"confusion_matrix_seed{seed}.png"
)

metrics_df_123.to_csv(metrics_path_123, index=False)
report_df_123.to_csv(report_path_123)

plot_confusion_matrix(
    confusion_df=confusion_df_123,
    output_path=confusion_path_123,
)

print("Seed 123 test metrics:")
display(metrics_df_123)

print("Saved metrics:", metrics_path_123)
print("Saved report:", report_path_123)
print("Saved predictions:", predictions_path_123)
print("Saved confusion matrix:", confusion_path_123)

Seed 123 test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall
0,0.703629,0.611103,0.732921,0.63406,0.801802


Saved metrics: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_seed123_metrics.csv
Saved report: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_seed123_classification_report.csv
Saved predictions: /kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_predictions_seed123.csv
Saved confusion matrix: /kaggle/working/skin-lesion-uncertainty/outputs/figures/confusion_matrix_seed123.png


In [29]:
# Phase 7:

seed = 123

seed123_files = [
    output_dirs["checkpoints_dir"]
    / f"model_seed{seed}.pth",

    output_dirs["tables_dir"]
    / f"training_history_seed{seed}.csv",

    output_dirs["tables_dir"]
    / f"baseline_seed{seed}_metrics.csv",

    output_dirs["tables_dir"]
    / f"baseline_seed{seed}_classification_report.csv",

    output_dirs["logits_dir"]
    / f"val_logits_seed{seed}.npy",

    output_dirs["logits_dir"]
    / f"val_labels_seed{seed}.npy",

    output_dirs["logits_dir"]
    / f"test_logits_seed{seed}.npy",

    output_dirs["logits_dir"]
    / f"test_labels_seed{seed}.npy",

    output_dirs["predictions_dir"]
    / f"test_predictions_seed{seed}.csv",

    output_dirs["figures_dir"]
    / f"training_curves_seed{seed}.png",

    output_dirs["figures_dir"]
    / f"confusion_matrix_seed{seed}.png",
]

all_seed123_present = True

for path in seed123_files:
    exists = path.exists()
    print(path.name, exists)
    all_seed123_present &= exists

print("\nSeed 123 complete:", all_seed123_present)

model_seed123.pth True
training_history_seed123.csv True
baseline_seed123_metrics.csv True
baseline_seed123_classification_report.csv True
val_logits_seed123.npy True
val_labels_seed123.npy True
test_logits_seed123.npy True
test_labels_seed123.npy True
test_predictions_seed123.csv True
training_curves_seed123.png True
confusion_matrix_seed123.png True

Seed 123 complete: True


In [30]:
# Phase 7: free up gpu memory

import gc
import torch

del model
del optimizer
del criterion

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GB",
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 3),
    "GB",
)

GPU memory allocated: 0.062 GB
GPU memory reserved: 0.131 GB


In [31]:
# Phase 7: Train seed 999  

seed = 999
set_global_seed(seed, deterministic=False)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=config["model"]["pretrained"],
    dropout_probability=config["model"]["dropout_probability"],
).to(device)

criterion = create_weighted_cross_entropy_loss(
    class_weights_df=class_weights_df,
    device=device,
)

optimizer = create_optimizer(
    model=model,
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
)

checkpoint_path = (
    output_dirs["checkpoints_dir"]
    / f"model_seed{seed}.pth"
)

history_path = (
    output_dirs["tables_dir"]
    / f"training_history_seed{seed}.csv"
)

print("Training seed:", seed)
print("Checkpoint:", checkpoint_path)
print("History:", history_path)



Training seed: 999
Checkpoint: /kaggle/working/skin-lesion-uncertainty/outputs/checkpoints/model_seed999.pth
History: /kaggle/working/skin-lesion-uncertainty/outputs/tables/training_history_seed999.csv


In [32]:
history_df_999 = train_baseline_model(
    model=model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    classes=classes,
    max_epochs=config["training"]["max_epochs"],
    patience=config["training"]["early_stopping_patience"],
    checkpoint_path=checkpoint_path,
    history_path=history_path,
    seed=seed,
    use_amp=config["training"]["mixed_precision"],
)

display(history_df_999)

print("Seed 999 training completed.")

/kaggle/working/skin-lesion-uncertainty/src/train.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


train:   0%|          | 0/251 [00:00<?, ?it/s]

/kaggle/working/skin-lesion-uncertainty/src/train.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
/kaggle/working/skin-lesion-uncertainty/src/train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


Epoch 1/12 - train_loss=1.7511 train_acc=0.5164 val_loss=1.0314 val_macro_f1=0.5387


/kaggle/working/skin-lesion-uncertainty/src/train.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


...

train:   0%|          | 0/251 [00:00<?, ?it/s]

/kaggle/working/skin-lesion-uncertainty/src/train.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f8004962fc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f8004962fc0>
Traceback (most recent

Epoch 9/12 - train_loss=0.3738 train_acc=0.8165 val_loss=0.6221 val_macro_f1=0.7154
Early stopping after 9 epochs.


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,val_melanoma_recall,best_val_macro_f1
0,1,1.751114,0.516411,1.031381,0.650495,0.472500,0.719551,0.538739,0.561983,0.538739
1,2,0.894096,0.677524,0.789103,0.738614,0.523124,0.725366,0.559870,0.421488,0.559870
2,3,0.686753,0.718707,0.784993,0.737624,0.542240,0.710363,0.595705,0.537190,0.595705
3,4,0.600109,0.744291,0.706638,0.735644,0.555246,0.757720,0.608724,0.694215,0.608724
4,5,0.516392,0.771621,0.788022,0.712871,0.560796,0.783771,0.626966,0.702479,0.626966
5,6,0.439393,0.802196,0.519485,0.800000,0.723203,0.780686,0.732917,0.694215,0.732917
6,7,0.476938,0.799950,0.740853,0.763366,0.621481,0.770749,0.665928,0.661157,0.732917
7,8,0.445297,0.809559,0.847047,0.740594,0.558811,0.789828,0.606097,0.652893,0.732917
8,9,0.373807,0.816548,0.622056,0.797030,0.649290,0.831610,0.715394,0.636364,0.732917


Seed 999 training completed.


In [37]:
# Phase 7: Save training curves

seed = 999

training_curve_path_999 = plot_training_curves(
    history_df=history_df_999,
    output_path=(
        output_dirs["figures_dir"]
        / f"training_curves_seed{seed}.png"
    ),
)

print("Saved:", training_curve_path_999)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/training_curves_seed999.png


In [38]:
# Phase 7: Load checkpoint and collect logits

checkpoint_999 = load_checkpoint(
    model=model,
    checkpoint_path=(
        output_dirs["checkpoints_dir"]
        / f"model_seed{seed}.pth"
    ),
    device=device,
)

print("Best checkpoint epoch:", checkpoint_999.get("epoch"))
print(
    "Best validation macro F1:",
    checkpoint_999.get("best_val_macro_f1"),
)

val_logits_999, val_labels_999, val_metadata_999 = collect_logits(
    model=model,
    dataloader=dataloaders["val"],
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

test_logits_999, test_labels_999, test_metadata_999 = collect_logits(
    model=model,
    dataloader=dataloaders["test"],
    device=device,
    use_amp=config["training"]["mixed_precision"],
)

save_logits_and_labels(
    logits=val_logits_999,
    labels=val_labels_999,
    output_dir=output_dirs["logits_dir"],
    split_name="val",
    seed=seed,
)

save_logits_and_labels(
    logits=test_logits_999,
    labels=test_labels_999,
    output_dir=output_dirs["logits_dir"],
    split_name="test",
    seed=seed,
)

print("Validation logits:", val_logits_999.shape)
print("Validation labels:", val_labels_999.shape)
print("Test logits:", test_logits_999.shape)
print("Test labels:", test_labels_999.shape)

Best checkpoint epoch: 6
Best validation macro F1: 0.7329170567151957


/kaggle/working/skin-lesion-uncertainty/src/inference.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


Validation logits: (1010, 7)
Validation labels: (1010,)
Test logits: (992, 7)
Test labels: (992,)


In [39]:
# Phase 7: Compute seed 999 metrics

test_metrics_999 = classification_metrics_from_logits(
    logits=test_logits_999,
    labels=test_labels_999,
    classes=classes,
)

predictions_df_999 = prediction_dataframe_from_logits(
    logits=test_logits_999,
    labels=test_labels_999,
    metadata_df=test_metadata_999,
    classes=classes,
)

predictions_path_999 = (
    output_dirs["predictions_dir"]
    / f"test_predictions_seed{seed}.csv"
)

predictions_df_999.to_csv(
    predictions_path_999,
    index=False,
)

predicted_labels_999 = (
    predictions_df_999["predicted_label"].to_numpy()
)

report_df_999 = classification_report_dataframe(
    labels=test_labels_999,
    predictions=predicted_labels_999,
    classes=classes,
)

confusion_df_999 = confusion_matrix_dataframe(
    labels=test_labels_999,
    predictions=predicted_labels_999,
    classes=classes,
)

metrics_df_999 = pd.DataFrame([test_metrics_999])

metrics_path_999 = (
    output_dirs["tables_dir"]
    / f"baseline_seed{seed}_metrics.csv"
)

report_path_999 = (
    output_dirs["tables_dir"]
    / f"baseline_seed{seed}_classification_report.csv"
)

confusion_path_999 = (
    output_dirs["figures_dir"]
    / f"confusion_matrix_seed{seed}.png"
)

metrics_df_999.to_csv(metrics_path_999, index=False)
report_df_999.to_csv(report_path_999)

plot_confusion_matrix(
    confusion_df=confusion_df_999,
    output_path=confusion_path_999,
)

print("Seed 999 test metrics:")
display(metrics_df_999)

print("Saved metrics:", metrics_path_999)
print("Saved report:", report_path_999)
print("Saved predictions:", predictions_path_999)
print("Saved confusion matrix:", confusion_path_999)


Seed 999 test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall
0,0.790323,0.71881,0.732688,0.714678,0.711712


Saved metrics: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_seed999_metrics.csv
Saved report: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_seed999_classification_report.csv
Saved predictions: /kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_predictions_seed999.csv
Saved confusion matrix: /kaggle/working/skin-lesion-uncertainty/outputs/figures/confusion_matrix_seed999.png


In [40]:
# Phase 7: Verify all three ensemble members

ensemble_seeds = [42, 123, 999]

missing_files = []

for ensemble_seed in ensemble_seeds:
    required_files = [
        output_dirs["checkpoints_dir"]
        / f"model_seed{ensemble_seed}.pth",

        output_dirs["tables_dir"]
        / f"training_history_seed{ensemble_seed}.csv",

        output_dirs["tables_dir"]
        / f"baseline_seed{ensemble_seed}_metrics.csv",

        output_dirs["logits_dir"]
        / f"val_logits_seed{ensemble_seed}.npy",

        output_dirs["logits_dir"]
        / f"val_labels_seed{ensemble_seed}.npy",

        output_dirs["logits_dir"]
        / f"test_logits_seed{ensemble_seed}.npy",

        output_dirs["logits_dir"]
        / f"test_labels_seed{ensemble_seed}.npy",
    ]

    print(f"\nSeed {ensemble_seed}")

    for path in required_files:
        exists = path.exists()
        print(path.name, exists)

        if not exists:
            missing_files.append(path)

if missing_files:
    print("\nMissing files:")
    for path in missing_files:
        print(path)
else:
    print("\nAll three ensemble members are complete.")


Seed 42
model_seed42.pth True
training_history_seed42.csv True
baseline_seed42_metrics.csv True
val_logits_seed42.npy True
val_labels_seed42.npy True
test_logits_seed42.npy True
test_labels_seed42.npy True

Seed 123
model_seed123.pth True
training_history_seed123.csv True
baseline_seed123_metrics.csv True
val_logits_seed123.npy True
val_labels_seed123.npy True
test_logits_seed123.npy True
test_labels_seed123.npy True

Seed 999
model_seed999.pth True
training_history_seed999.csv True
baseline_seed999_metrics.csv True
val_logits_seed999.npy True
val_labels_seed999.npy True
test_logits_seed999.npy True
test_labels_seed999.npy True

All three ensemble members are complete.


In [43]:
# Phase 7: Compare individual members

individual_metrics_frames = []

for ensemble_seed in [42, 123, 999]:
    metrics_path = (
        output_dirs["tables_dir"]
        / f"baseline_seed{ensemble_seed}_metrics.csv"
    )

    seed_metrics_df = pd.read_csv(metrics_path)
    seed_metrics_df.insert(0, "seed", ensemble_seed)
    individual_metrics_frames.append(seed_metrics_df)

individual_members_df = pd.concat(
    individual_metrics_frames,
    ignore_index=True,
)

individual_members_path = (
    output_dirs["tables_dir"]
    / "ensemble_individual_member_metrics.csv"
)

individual_members_df.to_csv(
    individual_members_path,
    index=False,
)

print("Individual ensemble-member metrics:")
display(individual_members_df)

print("Saved:", individual_members_path)


Individual ensemble-member metrics:


,seed,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall
0,42,0.812500,0.681854,0.757031,0.714404,0.675676
1,123,0.703629,0.611103,0.732921,0.634060,0.801802
2,999,0.790323,0.718810,0.732688,0.714678,0.711712


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/ensemble_individual_member_metrics.csv


In [44]:
# Phase 7: 

from pathlib import Path
import pandas as pd

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")

expected_files = [
    REPO_DIR / "outputs/checkpoints/model_seed123.pth",
    REPO_DIR / "outputs/checkpoints/model_seed999.pth",

    REPO_DIR / "outputs/tables/training_history_seed123.csv",
    REPO_DIR / "outputs/tables/training_history_seed999.csv",

    REPO_DIR / "outputs/tables/baseline_seed123_metrics.csv",
    REPO_DIR / "outputs/tables/baseline_seed999_metrics.csv",

    REPO_DIR / "outputs/tables/baseline_seed123_classification_report.csv",
    REPO_DIR / "outputs/tables/baseline_seed999_classification_report.csv",

    REPO_DIR / "outputs/logits/val_logits_seed123.npy",
    REPO_DIR / "outputs/logits/val_labels_seed123.npy",
    REPO_DIR / "outputs/logits/test_logits_seed123.npy",
    REPO_DIR / "outputs/logits/test_labels_seed123.npy",

    REPO_DIR / "outputs/logits/val_logits_seed999.npy",
    REPO_DIR / "outputs/logits/val_labels_seed999.npy",
    REPO_DIR / "outputs/logits/test_logits_seed999.npy",
    REPO_DIR / "outputs/logits/test_labels_seed999.npy",

    REPO_DIR / "outputs/predictions/test_predictions_seed123.csv",
    REPO_DIR / "outputs/predictions/test_predictions_seed999.csv",

    REPO_DIR / "outputs/figures/training_curves_seed123.png",
    REPO_DIR / "outputs/figures/training_curves_seed999.png",

    REPO_DIR / "outputs/figures/confusion_matrix_seed123.png",
    REPO_DIR / "outputs/figures/confusion_matrix_seed999.png",

    REPO_DIR / "outputs/tables/ensemble_individual_member_metrics.csv",
]

rows = []

for path in expected_files:
    exists = path.exists()
    size_bytes = path.stat().st_size if exists else 0

    rows.append({
        "file": str(path.relative_to(REPO_DIR)),
        "exists": exists,
        "non_empty": size_bytes > 0,
        "size_bytes": size_bytes,
    })

verification_df = pd.DataFrame(rows)

display(verification_df)

all_ok = (
    verification_df["exists"].all()
    and verification_df["non_empty"].all()
)

print("\nAll Phase 7 files exist and are non-empty:", all_ok)

if not all_ok:
    print("\nMissing or empty files:")
    display(
        verification_df[
            (~verification_df["exists"])
            | (~verification_df["non_empty"])
        ]
    )

,file,exists,non_empty,size_bytes
0,outputs/checkpoints/model_seed123.pth,True,True,48677759
1,outputs/checkpoints/model_seed999.pth,True,True,48677759
2,outputs/tables/training_history_seed123.csv,True,True,1177
3,outputs/tables/training_history_seed999.csv,True,True,1681
4,outputs/tables/baseline_seed123_metrics.csv,True,True,158
5,outputs/tables/baseline_seed999_metrics.csv,True,True,158
6,outputs/tables/baseline_seed123_classification...,True,True,675
7,outputs/tables/baseline_seed999_classification...,True,True,706
8,outputs/logits/val_logits_seed123.npy,True,True,14268
9,outputs/logits/val_labels_seed123.npy,True,True,8208



All Phase 7 files exist and are non-empty: True


In [45]:
# Phase 7: 
import numpy as np

array_files = [
    "outputs/logits/val_logits_seed123.npy",
    "outputs/logits/val_labels_seed123.npy",
    "outputs/logits/test_logits_seed123.npy",
    "outputs/logits/test_labels_seed123.npy",
    "outputs/logits/val_logits_seed999.npy",
    "outputs/logits/val_labels_seed999.npy",
    "outputs/logits/test_logits_seed999.npy",
    "outputs/logits/test_labels_seed999.npy",
]

for relative_path in array_files:
    path = REPO_DIR / relative_path
    array = np.load(path)
    print(relative_path, array.shape, array.dtype)

outputs/logits/val_logits_seed123.npy (1010, 7) float16
outputs/logits/val_labels_seed123.npy (1010,) int64
outputs/logits/test_logits_seed123.npy (992, 7) float16
outputs/logits/test_labels_seed123.npy (992,) int64
outputs/logits/val_logits_seed999.npy (1010, 7) float16
outputs/logits/val_labels_seed999.npy (1010,) int64
outputs/logits/test_logits_seed999.npy (992, 7) float16
outputs/logits/test_labels_seed999.npy (992,) int64


### Phase 8 — Deep Ensemble Aggregation, Calibration, Uncertainty, and Referral

In [46]:
# Phase 8: Back up completed Phase 7 outputs

from pathlib import Path
from datetime import datetime
import shutil

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_zip = shutil.make_archive(
    base_name=str(
        Path("/kaggle/working")
        / f"phase7_outputs_backup_{timestamp}"
    ),
    format="zip",
    root_dir=REPO_DIR,
    base_dir="outputs",
)

print("Backup created:", backup_zip)
print("Backup size:", Path(backup_zip).stat().st_size, "bytes")

Backup created: /kaggle/working/phase7_outputs_backup_20260725_121533.zip
Backup size: 136480835 bytes


In [47]:
# Phase 8: Pull the latest Phase 8 source code

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull", "--ff-only"], check=True)

print("Repository updated.")
print("Current directory:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-4"], check=True)

Updating 36b41d3..2ab83b7
Fast-forward
 README.md                     |   9 +-
 src/calibration.py            |  64 +++++++++++++
 src/metrics.py                |  60 ++++++++++++
 src/plots.py                  |  32 +++++++
 src/referral.py               |   2 +-
 src/uncertainty.py            | 126 +++++++++++++++++++++++++
 tests/test_phase8_ensemble.py | 210 ++++++++++++++++++++++++++++++++++++++++++
 7 files changed, 501 insertions(+), 2 deletions(-)
 create mode 100644 tests/test_phase8_ensemble.py
Repository updated.
Current directory: /kaggle/working/skin-lesion-uncertainty
2ab83b7 Phase 8 deep ensemble aggregation and uncertainty
36b41d3 Phase 6 uncertainty scoring and referral evaluation
f4a19f2 Phase 5 temperature scaling calibration
fccec8e Phase 4 baseline training utilities


From https://github.com/DiloJan-26/skin-lesion-uncertainty
   36b41d3..2ab83b7  main       -> origin/main


CompletedProcess(args=['git', 'log', '--oneline', '-4'], returncode=0)

In [48]:
# Phase 8: Reload updated modules

import importlib

import src.uncertainty
import src.calibration
import src.metrics
import src.referral
import src.plots

importlib.reload(src.uncertainty)
importlib.reload(src.calibration)
importlib.reload(src.metrics)
importlib.reload(src.referral)
importlib.reload(src.plots)

print("Phase 8 modules reloaded.")

Phase 8 modules reloaded.


In [49]:
# Phase 8: Import ensemble utilities

from src.uncertainty import (
    stack_member_probabilities,
    ensemble_mean_probabilities,
    ensemble_uncertainty_scores_dataframe,
    ensemble_error_detection_table,
)

from src.calibration import (
    fit_temperature_on_log_probabilities,
    apply_temperature_to_probabilities,
    save_temperature,
)

from src.metrics import (
    classification_metrics_from_probabilities,
    calibration_metrics_from_probabilities,
    reliability_curve_dataframe,
)

from src.referral import (
    referral_results_at_coverages,
    select_uncertainty_threshold_for_coverage,
    evaluate_referral_threshold,
    add_referral_decision,
)

from src.plots import (
    plot_reliability_diagram,
    plot_uncertainty_correct_vs_incorrect,
    plot_risk_coverage_curve,
    plot_model_comparison,
)

print("Phase 8 imports successful.")

Phase 8 imports successful.


In [50]:
# Phase 8: Load configuration and ensemble member logits

import numpy as np
import pandas as pd

from src.config import load_config
from src.paths import ensure_output_dirs

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

ensemble_seeds = [42, 123, 999]
classes = config["data"]["classes"]
coverages = config["referral"]["coverage_levels"]

val_member_logits = []
test_member_logits = []

for member_seed in ensemble_seeds:
    val_logits_path = (
        output_dirs["logits_dir"]
        / f"val_logits_seed{member_seed}.npy"
    )
    test_logits_path = (
        output_dirs["logits_dir"]
        / f"test_logits_seed{member_seed}.npy"
    )

    val_member_logits.append(np.load(val_logits_path))
    test_member_logits.append(np.load(test_logits_path))

val_labels = np.load(
    output_dirs["logits_dir"] / "val_labels_seed42.npy"
)
test_labels = np.load(
    output_dirs["logits_dir"] / "test_labels_seed42.npy"
)

for member_seed, val_logits, test_logits in zip(
    ensemble_seeds,
    val_member_logits,
    test_member_logits,
):
    print(
        f"Seed {member_seed}:",
        "val", val_logits.shape,
        "test", test_logits.shape,
    )

print("Validation labels:", val_labels.shape)
print("Test labels:", test_labels.shape)

Seed 42: val (1010, 7) test (992, 7)
Seed 123: val (1010, 7) test (992, 7)
Seed 999: val (1010, 7) test (992, 7)
Validation labels: (1010,)
Test labels: (992,)


In [51]:
# Phase 8: Verify labels are identical across ensemble members

for member_seed in ensemble_seeds:
    member_val_labels = np.load(
        output_dirs["logits_dir"]
        / f"val_labels_seed{member_seed}.npy"
    )
    member_test_labels = np.load(
        output_dirs["logits_dir"]
        / f"test_labels_seed{member_seed}.npy"
    )

    print(
        f"Seed {member_seed}:",
        "validation labels match =",
        np.array_equal(val_labels, member_val_labels),
        "| test labels match =",
        np.array_equal(test_labels, member_test_labels),
    )

Seed 42: validation labels match = True | test labels match = True
Seed 123: validation labels match = True | test labels match = True
Seed 999: validation labels match = True | test labels match = True


In [52]:
# Phase 8: Stack member probabilities and compute ensemble mean probabilities

val_member_probs = stack_member_probabilities(
    val_member_logits
)

test_member_probs = stack_member_probabilities(
    test_member_logits
)

val_ensemble_probs = ensemble_mean_probabilities(
    val_member_probs
)

test_ensemble_probs = ensemble_mean_probabilities(
    test_member_probs
)

print("Validation member probabilities:", val_member_probs.shape)
print("Test member probabilities:", test_member_probs.shape)
print("Validation ensemble probabilities:", val_ensemble_probs.shape)
print("Test ensemble probabilities:", test_ensemble_probs.shape)

print(
    "Validation probability sums valid:",
    np.allclose(val_ensemble_probs.sum(axis=1), 1.0),
)

print(
    "Test probability sums valid:",
    np.allclose(test_ensemble_probs.sum(axis=1), 1.0),
)

Validation member probabilities: (3, 1010, 7)
Test member probabilities: (3, 992, 7)
Validation ensemble probabilities: (1010, 7)
Test ensemble probabilities: (992, 7)
Validation probability sums valid: True
Test probability sums valid: True


In [53]:
# Phase 8: Save uncalibrated ensemble probabilities

val_ensemble_probs_path = (
    output_dirs["logits_dir"]
    / "val_probs_ensemble_uncalibrated.npy"
)

test_ensemble_probs_path = (
    output_dirs["logits_dir"]
    / "test_probs_ensemble_uncalibrated.npy"
)

np.save(val_ensemble_probs_path, val_ensemble_probs)
np.save(test_ensemble_probs_path, test_ensemble_probs)

print("Saved:", val_ensemble_probs_path)
print("Saved:", test_ensemble_probs_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/logits/val_probs_ensemble_uncalibrated.npy
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/logits/test_probs_ensemble_uncalibrated.npy


In [54]:
# Phase 8: Evaluate uncalibrated Deep Ensemble

ensemble_classification_uncalibrated = (
    classification_metrics_from_probabilities(
        probs=test_ensemble_probs,
        labels=test_labels,
        classes=classes,
    )
)

ensemble_calibration_uncalibrated = (
    calibration_metrics_from_probabilities(
        probs=test_ensemble_probs,
        labels=test_labels,
        classes=classes,
        n_bins=15,
    )
)

print("Uncalibrated ensemble classification metrics:")
display(pd.DataFrame([ensemble_classification_uncalibrated]))

print("Uncalibrated ensemble calibration metrics:")
display(pd.DataFrame([ensemble_calibration_uncalibrated]))

Uncalibrated ensemble classification metrics:


,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall
0,0.811492,0.721522,0.754754,0.730762,0.774775


Uncalibrated ensemble calibration metrics:


,accuracy,nll,brier_score,ece
0,0.811492,0.476446,0.257427,0.016339


In [55]:
# Phase 8: Fit ensemble Temperature Scaling on validation probabilities

import torch

ensemble_temperature = fit_temperature_on_log_probabilities(
    probabilities=val_ensemble_probs,
    labels=val_labels,
    max_iter=1000,
    lr=0.01,
    initial_temperature=1.0,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

ensemble_temperature_path = (
    output_dirs["tables_dir"]
    / "ensemble_temperature.csv"
)

save_temperature(
    ensemble_temperature,
    ensemble_temperature_path,
)

print("Learned ensemble temperature:", ensemble_temperature)
print("Saved:", ensemble_temperature_path)

Learned ensemble temperature: 1.0254298448562622
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/ensemble_temperature.csv


In [56]:
# Phase 8: Apply Temperature Scaling to ensemble probabilities

val_ensemble_probs_calibrated = (
    apply_temperature_to_probabilities(
        probabilities=val_ensemble_probs,
        temperature=ensemble_temperature,
    )
)

test_ensemble_probs_calibrated = (
    apply_temperature_to_probabilities(
        probabilities=test_ensemble_probs,
        temperature=ensemble_temperature,
    )
)

val_ensemble_calibrated_path = (
    output_dirs["logits_dir"]
    / "val_probs_ensemble_calibrated.npy"
)

test_ensemble_calibrated_path = (
    output_dirs["logits_dir"]
    / "test_probs_ensemble_calibrated.npy"
)

np.save(
    val_ensemble_calibrated_path,
    val_ensemble_probs_calibrated,
)

np.save(
    test_ensemble_calibrated_path,
    test_ensemble_probs_calibrated,
)

print(
    "Validation calibrated probabilities:",
    val_ensemble_probs_calibrated.shape,
)

print(
    "Test calibrated probabilities:",
    test_ensemble_probs_calibrated.shape,
)

print(
    "Test probability sums valid:",
    np.allclose(
        test_ensemble_probs_calibrated.sum(axis=1),
        1.0,
    ),
)

print("Saved:", val_ensemble_calibrated_path)
print("Saved:", test_ensemble_calibrated_path)

Validation calibrated probabilities: (1010, 7)
Test calibrated probabilities: (992, 7)
Test probability sums valid: True
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/logits/val_probs_ensemble_calibrated.npy
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/logits/test_probs_ensemble_calibrated.npy


In [57]:
# Phase 8: Evaluate calibrated Deep Ensemble

ensemble_classification_calibrated = (
    classification_metrics_from_probabilities(
        probs=test_ensemble_probs_calibrated,
        labels=test_labels,
        classes=classes,
    )
)

ensemble_calibration_calibrated = (
    calibration_metrics_from_probabilities(
        probs=test_ensemble_probs_calibrated,
        labels=test_labels,
        classes=classes,
        n_bins=15,
    )
)

ensemble_metrics_df = pd.DataFrame([
    {
        "model": "deep_ensemble_uncalibrated",
        "temperature": np.nan,
        **ensemble_classification_uncalibrated,
        **{
            "nll": ensemble_calibration_uncalibrated["nll"],
            "brier_score": ensemble_calibration_uncalibrated["brier_score"],
            "ece": ensemble_calibration_uncalibrated["ece"],
        },
    },
    {
        "model": "deep_ensemble_temperature_scaled",
        "temperature": ensemble_temperature,
        **ensemble_classification_calibrated,
        **{
            "nll": ensemble_calibration_calibrated["nll"],
            "brier_score": ensemble_calibration_calibrated["brier_score"],
            "ece": ensemble_calibration_calibrated["ece"],
        },
    },
])

ensemble_metrics_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_metrics.csv"
)

ensemble_metrics_df.to_csv(
    ensemble_metrics_path,
    index=False,
)

display(ensemble_metrics_df)
print("Saved:", ensemble_metrics_path)

,model,temperature,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall,nll,brier_score,ece
0,deep_ensemble_uncalibrated,NaN,0.811492,0.721522,0.754754,0.730762,0.774775,0.476446,0.257427,0.016339
1,deep_ensemble_temperature_scaled,1.02543,0.811492,0.721522,0.754754,0.730762,0.774775,0.475816,0.257357,0.020517


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/deep_ensemble_metrics.csv


In [58]:
# Phase 8: Create and save calibrated ensemble predictions

from src.inference import prediction_dataframe_from_probabilities

seed42_predictions_df = pd.read_csv(
    output_dirs["predictions_dir"]
    / "test_predictions_seed42.csv"
)

metadata_df = seed42_predictions_df[
    ["image_id", "lesion_id"]
].copy()

ensemble_predictions_df = (
    prediction_dataframe_from_probabilities(
        probs=test_ensemble_probs_calibrated,
        labels=test_labels,
        metadata_df=metadata_df,
        classes=classes,
    )
)

ensemble_predictions_path = (
    output_dirs["predictions_dir"]
    / "test_predictions_deep_ensemble_calibrated.csv"
)

ensemble_predictions_df.to_csv(
    ensemble_predictions_path,
    index=False,
)

display(ensemble_predictions_df.head())
print("Saved:", ensemble_predictions_path)

,image_id,lesion_id,true_label,true_class,predicted_label,predicted_class,confidence,correct,prob_akiec,prob_bcc,prob_bkl,prob_df,prob_mel,prob_nv,prob_vasc
0,ISIC_0027419,HAM_0000118,2,bkl,2,bkl,0.643803,True,0.167558,0.004691,0.643803,0.000511,0.170849,0.012427,0.000161
1,ISIC_0025030,HAM_0000118,2,bkl,2,bkl,0.729030,True,0.133759,0.007146,0.729030,0.000764,0.119048,0.009981,0.000272
2,ISIC_0029176,HAM_0002761,2,bkl,2,bkl,0.554935,True,0.178997,0.045206,0.554935,0.039442,0.010946,0.166571,0.003903
3,ISIC_0029068,HAM_0002761,2,bkl,2,bkl,0.389793,True,0.077451,0.358753,0.389793,0.024126,0.030880,0.117700,0.001297
4,ISIC_0032343,HAM_0006071,2,bkl,2,bkl,0.898532,True,0.003490,0.000007,0.898532,0.000072,0.094228,0.003666,0.000005


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_predictions_deep_ensemble_calibrated.csv


In [59]:
# Phase 8: Generate ensemble uncertainty scores

ensemble_scores_df = ensemble_uncertainty_scores_dataframe(
    member_probabilities=test_member_probs,
    labels=test_labels,
    classes=classes,
    metadata_df=metadata_df,
)

ensemble_scores_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_uncertainty_scores.csv"
)

ensemble_scores_df.to_csv(
    ensemble_scores_path,
    index=False,
)

display(ensemble_scores_df.head())

print("\nEnsemble uncertainty summary:")
display(
    ensemble_scores_df[
        [
            "confidence",
            "confidence_uncertainty",
            "predictive_entropy",
            "expected_entropy",
            "mutual_information",
            "probability_variance",
            "correct",
        ]
    ].describe()
)

print("Saved:", ensemble_scores_path)

,image_id,lesion_id,true_label,true_class,predicted_label,predicted_class,confidence,confidence_uncertainty,predictive_entropy,expected_entropy,mutual_information,probability_variance,correct
0,ISIC_0027419,HAM_0000118,2,bkl,2,bkl,0.652388,0.347612,0.952854,0.621466,0.331388,0.026065,True
1,ISIC_0025030,HAM_0000118,2,bkl,2,bkl,0.738635,0.261365,0.819520,0.596532,0.222988,0.015388,True
2,ISIC_0029176,HAM_0002761,2,bkl,2,bkl,0.564507,0.435493,1.250880,0.919349,0.331531,0.023550,True
3,ISIC_0029068,HAM_0002761,2,bkl,2,bkl,0.394190,0.605810,1.375682,1.182845,0.192838,0.010252,True
4,ISIC_0032343,HAM_0006071,2,bkl,2,bkl,0.904145,0.095855,0.343912,0.332259,0.011653,0.000364,True



Ensemble uncertainty summary:


,confidence,confidence_uncertainty,predictive_entropy,expected_entropy,mutual_information,probability_variance
count,992.000000,9.920000e+02,9.920000e+02,9.920000e+02,992.000000,9.920000e+02
mean,0.815350,1.846499e-01,4.464650e-01,3.219170e-01,0.124548,1.024235e-02
std,0.201707,2.017073e-01,4.336443e-01,3.247426e-01,0.160067,1.470567e-02
min,0.245077,3.330669e-16,1.657865e-10,1.657863e-10,0.000000,1.215886e-32
25%,0.644691,1.441620e-03,1.142396e-02,1.002966e-02,0.001189,6.738745e-07
50%,0.908902,9.109821e-02,3.506659e-01,2.383908e-01,0.040930,1.247052e-03
75%,0.998558,3.553086e-01,7.917451e-01,5.690537e-01,0.208497,1.663389e-02
max,1.000000,7.549231e-01,1.760056e+00,1.418719e+00,0.849638,7.529862e-02


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/deep_ensemble_uncertainty_scores.csv


In [60]:
# Phase 8: Compute ensemble error-detection AUROC

ensemble_error_detection_df = ensemble_error_detection_table(
    ensemble_scores_df
)

ensemble_error_detection_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_error_detection.csv"
)

ensemble_error_detection_df.to_csv(
    ensemble_error_detection_path,
    index=False,
)

display(ensemble_error_detection_df)
print("Saved:", ensemble_error_detection_path)

,score,error_detection_auroc
0,confidence_uncertainty,0.867792
1,predictive_entropy,0.867752
2,mutual_information,0.796964
3,probability_variance,0.804092


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/deep_ensemble_error_detection.csv


In [61]:
# Phase 8: Compute ensemble referral results at fixed coverage levels

ensemble_referral_frames = []

ensemble_uncertainty_methods = {
    "ensemble_confidence_uncertainty":
        ensemble_scores_df["confidence_uncertainty"].to_numpy(),

    "ensemble_predictive_entropy":
        ensemble_scores_df["predictive_entropy"].to_numpy(),

    "ensemble_mutual_information":
        ensemble_scores_df["mutual_information"].to_numpy(),

    "ensemble_probability_variance":
        ensemble_scores_df["probability_variance"].to_numpy(),
}

for method_name, uncertainty_values in (
    ensemble_uncertainty_methods.items()
):
    method_results_df = referral_results_at_coverages(
        probs=test_ensemble_probs_calibrated,
        labels=test_labels,
        uncertainty_scores=uncertainty_values,
        coverages=coverages,
        classes=classes,
        method_name=method_name,
    )

    ensemble_referral_frames.append(method_results_df)

ensemble_referral_df = pd.concat(
    ensemble_referral_frames,
    ignore_index=True,
)

ensemble_referral_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_referral_results.csv"
)

ensemble_referral_df.to_csv(
    ensemble_referral_path,
    index=False,
)

display(ensemble_referral_df)
print("Saved:", ensemble_referral_path)

,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1
0,ensemble_confidence_uncertainty,1.000000,0.000000,992,0,0.811492,0.188508,0.730762
1,ensemble_confidence_uncertainty,0.950605,0.049395,943,49,0.835631,0.164369,0.779449
2,ensemble_confidence_uncertainty,0.900202,0.099798,893,99,0.853303,0.146697,0.808020
3,ensemble_confidence_uncertainty,0.850806,0.149194,844,148,0.876777,0.123223,0.825962
4,ensemble_confidence_uncertainty,0.800403,0.199597,794,198,0.896725,0.103275,0.852330
5,ensemble_confidence_uncertainty,0.700605,0.299395,695,297,0.933813,0.066187,0.896966
6,ensemble_confidence_uncertainty,0.600806,0.399194,596,396,0.963087,0.036913,0.945587
7,ensemble_predictive_entropy,1.000000,0.000000,992,0,0.811492,0.188508,0.730762
8,ensemble_predictive_entropy,0.950605,0.049395,943,49,0.834571,0.165429,0.776161
9,ensemble_predictive_entropy,0.900202,0.099798,893,99,0.853303,0.146697,0.804513


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/deep_ensemble_referral_results.csv


In [62]:
# Phase 8: Select ensemble referral threshold on validation data

val_ensemble_confidence_uncertainty = (
    1.0 - np.max(val_ensemble_probs_calibrated, axis=1)
)

test_ensemble_confidence_uncertainty = (
    1.0 - np.max(test_ensemble_probs_calibrated, axis=1)
)

target_coverage = 0.90

ensemble_selected_threshold = (
    select_uncertainty_threshold_for_coverage(
        uncertainty_scores=val_ensemble_confidence_uncertainty,
        target_coverage=target_coverage,
    )
)

ensemble_selected_result = evaluate_referral_threshold(
    probs=test_ensemble_probs_calibrated,
    labels=test_labels,
    uncertainty_scores=test_ensemble_confidence_uncertainty,
    threshold=ensemble_selected_threshold,
    classes=classes,
    method_name=(
        "validation_selected_ensemble_confidence_uncertainty"
    ),
)

ensemble_threshold_df = pd.DataFrame([{
    "method": "ensemble_confidence_uncertainty",
    "target_validation_coverage": target_coverage,
    "selected_threshold": ensemble_selected_threshold,
}])

ensemble_selected_result_df = pd.DataFrame([
    ensemble_selected_result
])

ensemble_threshold_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_selected_referral_threshold.csv"
)

ensemble_selected_result_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_selected_referral_test_result.csv"
)

ensemble_threshold_df.to_csv(
    ensemble_threshold_path,
    index=False,
)

ensemble_selected_result_df.to_csv(
    ensemble_selected_result_path,
    index=False,
)

display(ensemble_threshold_df)
display(ensemble_selected_result_df)

print("Saved:", ensemble_threshold_path)
print("Saved:", ensemble_selected_result_path)

,method,target_validation_coverage,selected_threshold
0,ensemble_confidence_uncertainty,0.9,0.483972


,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1,threshold
0,validation_selected_ensemble_confidence_uncert...,0.889113,0.110887,882,110,0.860544,0.139456,0.814715,0.483972


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/deep_ensemble_selected_referral_threshold.csv
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/deep_ensemble_selected_referral_test_result.csv


In [63]:
# Phase 8: Save ensemble accept or refer decisions

ensemble_decision_df = ensemble_predictions_df.copy()

ensemble_decision_df[
    "confidence_uncertainty"
] = test_ensemble_confidence_uncertainty

ensemble_decision_df = add_referral_decision(
    scores_df=ensemble_decision_df,
    uncertainty_column="confidence_uncertainty",
    threshold=ensemble_selected_threshold,
)

ensemble_decision_path = (
    output_dirs["predictions_dir"]
    / "test_referral_decisions_deep_ensemble.csv"
)

ensemble_decision_df.to_csv(
    ensemble_decision_path,
    index=False,
)

display(
    ensemble_decision_df["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="count")
)

print("Saved:", ensemble_decision_path)

,decision,count
0,accept,882
1,refer,110


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_referral_decisions_deep_ensemble.csv


In [64]:
# Phase 8: Create ensemble reliability diagram

ensemble_uncalibrated_curve_df = (
    reliability_curve_dataframe(
        probs=test_ensemble_probs,
        labels=test_labels,
        n_bins=15,
    )
)

ensemble_calibrated_curve_df = (
    reliability_curve_dataframe(
        probs=test_ensemble_probs_calibrated,
        labels=test_labels,
        n_bins=15,
    )
)

ensemble_uncalibrated_curve_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_reliability_uncalibrated.csv"
)

ensemble_calibrated_curve_path = (
    output_dirs["tables_dir"]
    / "deep_ensemble_reliability_calibrated.csv"
)

ensemble_uncalibrated_curve_df.to_csv(
    ensemble_uncalibrated_curve_path,
    index=False,
)

ensemble_calibrated_curve_df.to_csv(
    ensemble_calibrated_curve_path,
    index=False,
)

ensemble_reliability_plot_path = plot_reliability_diagram(
    uncalibrated_curve_df=ensemble_uncalibrated_curve_df,
    calibrated_curve_df=ensemble_calibrated_curve_df,
    output_path=(
        output_dirs["figures_dir"]
        / "deep_ensemble_reliability_diagram.png"
    ),
)

print("Saved:", ensemble_reliability_plot_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/deep_ensemble_reliability_diagram.png


In [65]:
# Phase 8: Create ensemble uncertainty plot

ensemble_uncertainty_plot_path = (
    plot_uncertainty_correct_vs_incorrect(
        scores_df=ensemble_scores_df,
        score_column="mutual_information",
        output_path=(
            output_dirs["figures_dir"]
            / "deep_ensemble_uncertainty_correct_vs_incorrect.png"
        ),
    )
)

print("Saved:", ensemble_uncertainty_plot_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/deep_ensemble_uncertainty_correct_vs_incorrect.png


In [66]:
# Phase 8: Create ensemble risk-coverage curve

ensemble_risk_coverage_plot_path = (
    plot_risk_coverage_curve(
        referral_results_df=ensemble_referral_df,
        output_path=(
            output_dirs["figures_dir"]
            / "deep_ensemble_risk_coverage_curve.png"
        ),
    )
)

print("Saved:", ensemble_risk_coverage_plot_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/deep_ensemble_risk_coverage_curve.png


In [67]:
# Phase 8: Build baseline versus Deep Ensemble comparison table

baseline_seed42_metrics_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "baseline_seed42_metrics.csv"
)

baseline_calibration_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "calibration_metrics_seed42.csv"
)

baseline_classification = (
    baseline_seed42_metrics_df.iloc[0].to_dict()
)

baseline_uncalibrated_calibration = (
    baseline_calibration_df[
        baseline_calibration_df["model"]
        == "baseline_uncalibrated"
    ].iloc[0].to_dict()
)

baseline_calibrated_calibration = (
    baseline_calibration_df[
        baseline_calibration_df["model"]
        == "baseline_temperature_scaled"
    ].iloc[0].to_dict()
)

model_comparison_df = pd.DataFrame([
    {
        "model": "single_seed42_uncalibrated",
        "accuracy": baseline_classification["accuracy"],
        "macro_f1": baseline_classification["macro_f1"],
        "melanoma_recall":
            baseline_classification["melanoma_recall"],
        "nll": baseline_uncalibrated_calibration["nll"],
        "brier_score":
            baseline_uncalibrated_calibration["brier_score"],
        "ece": baseline_uncalibrated_calibration["ece"],
    },
    {
        "model": "single_seed42_temperature_scaled",
        "accuracy": baseline_classification["accuracy"],
        "macro_f1": baseline_classification["macro_f1"],
        "melanoma_recall":
            baseline_classification["melanoma_recall"],
        "nll": baseline_calibrated_calibration["nll"],
        "brier_score":
            baseline_calibrated_calibration["brier_score"],
        "ece": baseline_calibrated_calibration["ece"],
    },
    {
        "model": "deep_ensemble_uncalibrated",
        "accuracy":
            ensemble_classification_uncalibrated["accuracy"],
        "macro_f1":
            ensemble_classification_uncalibrated["macro_f1"],
        "melanoma_recall":
            ensemble_classification_uncalibrated[
                "melanoma_recall"
            ],
        "nll": ensemble_calibration_uncalibrated["nll"],
        "brier_score":
            ensemble_calibration_uncalibrated["brier_score"],
        "ece": ensemble_calibration_uncalibrated["ece"],
    },
    {
        "model": "deep_ensemble_temperature_scaled",
        "accuracy":
            ensemble_classification_calibrated["accuracy"],
        "macro_f1":
            ensemble_classification_calibrated["macro_f1"],
        "melanoma_recall":
            ensemble_classification_calibrated[
                "melanoma_recall"
            ],
        "nll": ensemble_calibration_calibrated["nll"],
        "brier_score":
            ensemble_calibration_calibrated["brier_score"],
        "ece": ensemble_calibration_calibrated["ece"],
    },
])

model_comparison_path = (
    output_dirs["tables_dir"]
    / "baseline_vs_deep_ensemble_comparison.csv"
)

model_comparison_df.to_csv(
    model_comparison_path,
    index=False,
)

display(model_comparison_df)
print("Saved:", model_comparison_path)

,model,accuracy,macro_f1,melanoma_recall,nll,brier_score,ece
0,single_seed42_uncalibrated,0.812500,0.714404,0.675676,0.600292,0.280098,0.082272
1,single_seed42_temperature_scaled,0.812500,0.714404,0.675676,0.505970,0.264806,0.030612
2,deep_ensemble_uncalibrated,0.811492,0.730762,0.774775,0.476446,0.257427,0.016339
3,deep_ensemble_temperature_scaled,0.811492,0.730762,0.774775,0.475816,0.257357,0.020517


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/baseline_vs_deep_ensemble_comparison.csv


In [68]:
# Phase 8: Plot classification model comparison

classification_comparison_plot = plot_model_comparison(
    comparison_df=model_comparison_df,
    metric_columns=[
        "accuracy",
        "macro_f1",
        "melanoma_recall",
    ],
    output_path=(
        output_dirs["figures_dir"]
        / "baseline_vs_deep_ensemble_classification.png"
    ),
)

print("Saved:", classification_comparison_plot)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/baseline_vs_deep_ensemble_classification.png


In [69]:
# Phase 8: Plot calibration model comparison

calibration_comparison_plot = plot_model_comparison(
    comparison_df=model_comparison_df,
    metric_columns=[
        "nll",
        "brier_score",
        "ece",
    ],
    output_path=(
        output_dirs["figures_dir"]
        / "baseline_vs_deep_ensemble_calibration.png"
    ),
)

print("Saved:", calibration_comparison_plot)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/baseline_vs_deep_ensemble_calibration.png


In [70]:
# Phase 8: Verify all generated Phase 8 files

phase8_expected_files = [
    output_dirs["logits_dir"]
    / "val_probs_ensemble_uncalibrated.npy",

    output_dirs["logits_dir"]
    / "test_probs_ensemble_uncalibrated.npy",

    output_dirs["logits_dir"]
    / "val_probs_ensemble_calibrated.npy",

    output_dirs["logits_dir"]
    / "test_probs_ensemble_calibrated.npy",

    output_dirs["tables_dir"]
    / "ensemble_temperature.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_metrics.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_uncertainty_scores.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_error_detection.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_referral_results.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_selected_referral_threshold.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_selected_referral_test_result.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_reliability_uncalibrated.csv",

    output_dirs["tables_dir"]
    / "deep_ensemble_reliability_calibrated.csv",

    output_dirs["tables_dir"]
    / "baseline_vs_deep_ensemble_comparison.csv",

    output_dirs["predictions_dir"]
    / "test_predictions_deep_ensemble_calibrated.csv",

    output_dirs["predictions_dir"]
    / "test_referral_decisions_deep_ensemble.csv",

    output_dirs["figures_dir"]
    / "deep_ensemble_reliability_diagram.png",

    output_dirs["figures_dir"]
    / "deep_ensemble_uncertainty_correct_vs_incorrect.png",

    output_dirs["figures_dir"]
    / "deep_ensemble_risk_coverage_curve.png",

    output_dirs["figures_dir"]
    / "baseline_vs_deep_ensemble_classification.png",

    output_dirs["figures_dir"]
    / "baseline_vs_deep_ensemble_calibration.png",
]

verification_rows = []

for path in phase8_expected_files:
    exists = path.exists()
    size_bytes = path.stat().st_size if exists else 0

    verification_rows.append({
        "file": str(path.relative_to(REPO_DIR)),
        "exists": exists,
        "non_empty": size_bytes > 0,
        "size_bytes": size_bytes,
    })

phase8_verification_df = pd.DataFrame(
    verification_rows
)

display(phase8_verification_df)

phase8_all_ok = (
    phase8_verification_df["exists"].all()
    and phase8_verification_df["non_empty"].all()
)

print(
    "\nAll Phase 8 files exist and are non-empty:",
    phase8_all_ok,
)

if not phase8_all_ok:
    print("\nMissing or empty files:")
    display(
        phase8_verification_df[
            (~phase8_verification_df["exists"])
            | (~phase8_verification_df["non_empty"])
        ]
    )

,file,exists,non_empty,size_bytes
0,outputs/logits/val_probs_ensemble_uncalibrated...,True,True,56688
1,outputs/logits/test_probs_ensemble_uncalibrate...,True,True,55680
2,outputs/logits/val_probs_ensemble_calibrated.npy,True,True,56688
3,outputs/logits/test_probs_ensemble_calibrated.npy,True,True,55680
4,outputs/tables/ensemble_temperature.csv,True,True,40
5,outputs/tables/deep_ensemble_metrics.csv,True,True,486
6,outputs/tables/deep_ensemble_uncertainty_score...,True,True,161381
7,outputs/tables/deep_ensemble_error_detection.csv,True,True,185
8,outputs/tables/deep_ensemble_referral_results.csv,True,True,3702
9,outputs/tables/deep_ensemble_selected_referral...,True,True,109



All Phase 8 files exist and are non-empty: True


In [71]:
# Phase 8: Back up completed Phase 8 outputs

from pathlib import Path
from datetime import datetime
import shutil

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_zip = shutil.make_archive(
    base_name=str(
        Path("/kaggle/working")
        / f"phase8_outputs_backup_{timestamp}"
    ),
    format="zip",
    root_dir=REPO_DIR,
    base_dir="outputs",
)

print("Backup created:", backup_zip)
print("Backup size:", Path(backup_zip).stat().st_size, "bytes")

Backup created: /kaggle/working/phase8_outputs_backup_20260725_123809.zip
Backup size: 137254180 bytes


### Phase 9 — MC Dropout and Final Uncertainty Comparison

In [74]:
# Phase 9: Pull the latest Phase 9 source code

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull", "--ff-only"], check=True)

print("Repository updated.")
subprocess.run(["git", "log", "--oneline", "-4"], check=True)

Updating 2ab83b7..b8ae718
Fast-forward
 README.md                       |   8 +-
 src/inference.py                | 101 +++++++++++++++++++++++
 src/metrics.py                  |   2 +-
 src/plots.py                    |  32 ++++++++
 src/uncertainty.py              |  20 +++++
 tests/test_phase9_mc_dropout.py | 176 ++++++++++++++++++++++++++++++++++++++++
 6 files changed, 337 insertions(+), 2 deletions(-)
 create mode 100644 tests/test_phase9_mc_dropout.py
Repository updated.
b8ae718 Phase 9 MC Dropout uncertainty comparison
2ab83b7 Phase 8 deep ensemble aggregation and uncertainty
36b41d3 Phase 6 uncertainty scoring and referral evaluation
f4a19f2 Phase 5 temperature scaling calibration


From https://github.com/DiloJan-26/skin-lesion-uncertainty
   2ab83b7..b8ae718  main       -> origin/main


CompletedProcess(args=['git', 'log', '--oneline', '-4'], returncode=0)

In [77]:
# Phase 9: Reload updated modules

import importlib

import src.inference
import src.uncertainty
import src.metrics
import src.plots

importlib.reload(src.inference)
importlib.reload(src.uncertainty)
importlib.reload(src.metrics)
importlib.reload(src.plots)

print("Phase 9 modules reloaded.")

Phase 9 modules reloaded.


In [78]:
# Phase 9: Import MC Dropout utilities

from src.inference import (
    load_checkpoint,
    collect_mc_dropout_probabilities,
)

from src.uncertainty import (
    ensemble_mean_probabilities,
    mc_dropout_uncertainty_scores_dataframe,
    mc_dropout_error_detection_table,
)

from src.calibration import (
    fit_temperature_on_log_probabilities,
    apply_temperature_to_probabilities,
    save_temperature,
)

from src.metrics import (
    classification_metrics_from_probabilities,
    calibration_metrics_from_probabilities,
    reliability_curve_dataframe,
)

from src.referral import (
    referral_results_at_coverages,
    select_uncertainty_threshold_for_coverage,
    evaluate_referral_threshold,
    add_referral_decision,
)

from src.plots import (
    plot_reliability_diagram,
    plot_uncertainty_correct_vs_incorrect,
    plot_risk_coverage_curve,
    plot_model_comparison,
    plot_uncertainty_method_comparison,
)

print("Phase 9 imports successful.")

Phase 9 imports successful.


In [79]:
# Phase 9: Rebuild dataloaders and seed-42 model

import torch
import pandas as pd
import numpy as np

from src.config import load_config
from src.paths import ensure_output_dirs
from src.data import build_datasets_and_loaders
from src.models import create_efficientnet_classifier

config = load_config("configs/experiment.yaml")
output_dirs = ensure_output_dirs(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classes = config["data"]["classes"]
coverages = config["referral"]["coverage_levels"]

datasets, dataloaders, class_weights_df = build_datasets_and_loaders(
    split_dir=output_dirs["tables_dir"],
    classes=classes,
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    fallback_batch_size=config["training"]["fallback_batch_size"],
    num_workers=2,
)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=False,
    dropout_probability=config["model"]["dropout_probability"],
).to(device)

checkpoint_path = (
    output_dirs["checkpoints_dir"] / "model_seed42.pth"
)

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

print("Device:", device)
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Best validation macro F1:", checkpoint.get("best_val_macro_f1"))
print("Validation samples:", len(datasets["val"]))
print("Test samples:", len(datasets["test"]))

Device: cuda
Checkpoint epoch: 9
Best validation macro F1: 0.7606708798678457
Validation samples: 1010
Test samples: 992


In [104]:
# Phase 9 correction: Pull corrected MC Dropout implementation

from pathlib import Path
import os
import subprocess

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
os.chdir(REPO_DIR)

subprocess.run(["git", "pull", "--ff-only"], check=True)

print("Repository updated.")
subprocess.run(["git", "log", "--oneline", "-4"], check=True)

Updating b8ae718..4fbd7c2
Fast-forward
 README.md                             |  14 ++++
 src/inference.py                      |  48 +++++++++++--
 src/models.py                         |  60 +++++++++++++++-
 tests/test_mc_dropout_construction.py | 126 ++++++++++++++++++++++++++++++++++
 4 files changed, 238 insertions(+), 10 deletions(-)
 create mode 100644 tests/test_mc_dropout_construction.py
Repository updated.
4fbd7c2 Fix MC Dropout stochastic inference
b8ae718 Phase 9 MC Dropout uncertainty comparison
2ab83b7 Phase 8 deep ensemble aggregation and uncertainty
36b41d3 Phase 6 uncertainty scoring and referral evaluation


From https://github.com/DiloJan-26/skin-lesion-uncertainty
   b8ae718..4fbd7c2  main       -> origin/main


CompletedProcess(args=['git', 'log', '--oneline', '-4'], returncode=0)

### Phase 9 — Corrected MC dropout evaluation

In [105]:
# Phase 9 correction: Reload corrected modules

import importlib

import src.models
import src.inference
import src.uncertainty
import src.metrics
import src.calibration
import src.referral
import src.plots

importlib.reload(src.models)
importlib.reload(src.inference)
importlib.reload(src.uncertainty)
importlib.reload(src.metrics)
importlib.reload(src.calibration)
importlib.reload(src.referral)
importlib.reload(src.plots)

print("Corrected Phase 9 modules reloaded.")

Corrected Phase 9 modules reloaded.


In [106]:
# Phase 9 correction: Import corrected MC Dropout utilities

from src.models import create_efficientnet_classifier

from src.inference import (
    load_checkpoint,
    inspect_dropout_modules,
    enable_mc_dropout,
    collect_mc_dropout_probabilities,
    prediction_dataframe_from_probabilities,
)

from src.uncertainty import (
    ensemble_mean_probabilities,
    mc_dropout_uncertainty_scores_dataframe,
    mc_dropout_error_detection_table,
)

from src.calibration import (
    fit_temperature_on_log_probabilities,
    apply_temperature_to_probabilities,
    save_temperature,
)

from src.metrics import (
    classification_metrics_from_probabilities,
    calibration_metrics_from_probabilities,
    reliability_curve_dataframe,
)

from src.referral import (
    referral_results_at_coverages,
    select_uncertainty_threshold_for_coverage,
    evaluate_referral_threshold,
    add_referral_decision,
)

from src.plots import (
    plot_reliability_diagram,
    plot_uncertainty_correct_vs_incorrect,
    plot_risk_coverage_curve,
    plot_model_comparison,
)

print("Corrected Phase 9 imports successful.")

Corrected Phase 9 imports successful.


In [107]:
# Phase 9 correction: Recreate corrected model and load seed-42 checkpoint

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = create_efficientnet_classifier(
    backbone=config["model"]["main_backbone"],
    num_classes=config["model"]["num_classes"],
    pretrained=False,
    dropout_probability=config["model"]["dropout_probability"],
).to(device)

checkpoint_path = (
    output_dirs["checkpoints_dir"]
    / "model_seed42.pth"
)

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

print("Device:", device)
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Best validation macro F1:", checkpoint.get("best_val_macro_f1"))
print("Configured dropout:", config["model"]["dropout_probability"])

Device: cuda
Checkpoint epoch: 9
Best validation macro F1: 0.7606708798678457
Configured dropout: 0.2


In [108]:
# Phase 9 correction: Inspect registered dropout modules

dropout_info = inspect_dropout_modules(model)

print("Number of registered dropout modules:", len(dropout_info))

for item in dropout_info:
    print(item)

Number of registered dropout modules: 1
Dropout(p=0.2, inplace=False)


In [109]:
# Phase 9 correction: Verify Dropout activates and BatchNorm remains frozen

enable_mc_dropout(model)

active_dropout_count = 0
training_batchnorm_count = 0

for name, module in model.named_modules():
    if isinstance(
        module,
        (
            torch.nn.Dropout,
            torch.nn.Dropout1d,
            torch.nn.Dropout2d,
            torch.nn.Dropout3d,
            torch.nn.AlphaDropout,
        ),
    ):
        print(
            "Dropout:",
            name,
            "p=",
            module.p,
            "training=",
            module.training,
        )

        if module.training and module.p > 0:
            active_dropout_count += 1

    if isinstance(
        module,
        torch.nn.modules.batchnorm._BatchNorm,
    ):
        if module.training:
            training_batchnorm_count += 1

print("Active dropout modules:", active_dropout_count)
print("BatchNorm modules in training mode:", training_batchnorm_count)

Dropout: classifier.dropout p= 0.2 training= True
Active dropout modules: 1
BatchNorm modules in training mode: 0


In [111]:
# Phase 9 correction: Inspect one test batch structure

batch = next(iter(dataloaders["test"]))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("Batch keys:", batch.keys())

    for key, value in batch.items():
        if hasattr(value, "shape"):
            print(key, value.shape, value.dtype)
        else:
            print(key, type(value))
else:
    print("Batch length:", len(batch))

Batch type: <class 'dict'>
Batch keys: dict_keys(['image', 'label', 'image_id', 'lesion_id', 'class_name'])
image torch.Size([32, 3, 224, 224]) torch.float32
label torch.Size([32]) torch.int64
image_id <class 'list'>
lesion_id <class 'list'>
class_name <class 'list'>


In [112]:
# Phase 9 correction: Run stochastic smoke test on one test batch

import numpy as np
import torch

batch = next(iter(dataloaders["test"]))

if isinstance(batch, dict):
    if "image" in batch:
        images = batch["image"]
    elif "images" in batch:
        images = batch["images"]
    else:
        raise KeyError(
            f"Could not find image tensor. Available keys: {list(batch.keys())}"
        )
else:
    images = batch[0]

images = images.to(device)

smoke_probabilities = []

with torch.no_grad():
    for pass_index in range(3):
        enable_mc_dropout(model)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(
                config["training"]["mixed_precision"]
                and device.type == "cuda"
            ),
        ):
            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)

        smoke_probabilities.append(
            probabilities.detach().cpu().numpy()
        )

smoke_probabilities = np.stack(
    smoke_probabilities,
    axis=0,
)

maximum_difference = np.max(
    np.abs(
        smoke_probabilities[0]
        - smoke_probabilities[1]
    )
)

mean_variance = np.var(
    smoke_probabilities,
    axis=0,
).mean()

all_identical = np.allclose(
    smoke_probabilities,
    smoke_probabilities[0][None, :, :],
)

print("Smoke-test shape:", smoke_probabilities.shape)
print("Maximum pass difference:", maximum_difference)
print("Mean probability variance:", mean_variance)
print("All passes identical:", all_identical)

Smoke-test shape: (3, 32, 7)
Maximum pass difference: 0.37185314
Mean probability variance: 0.00202311
All passes identical: False


In [113]:
# Phase 9 correction: Run 20-pass MC Dropout on validation data

num_mc_passes = 20

val_mc_probabilities, val_mc_labels, val_mc_metadata = (
    collect_mc_dropout_probabilities(
        model=model,
        dataloader=dataloaders["val"],
        device=device,
        num_passes=num_mc_passes,
        use_amp=config["training"]["mixed_precision"],
    )
)

print(
    "Validation MC probabilities:",
    val_mc_probabilities.shape,
)

print(
    "Validation labels:",
    val_mc_labels.shape,
)

/kaggle/working/skin-lesion-uncertainty/src/inference.py:247: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Validation MC probabilities: (20, 1010, 7)
Validation labels: (1010,)


In [115]:
# Phase 9 correction: Run 20-pass MC Dropout on test data

test_mc_probabilities, test_mc_labels, test_mc_metadata = (
    collect_mc_dropout_probabilities(
        model=model,
        dataloader=dataloaders["test"],
        device=device,
        num_passes=num_mc_passes,
        use_amp=config["training"]["mixed_precision"],
    )
)

print(
    "Test MC probabilities:",
    test_mc_probabilities.shape,
)

print(
    "Test labels:",
    test_mc_labels.shape,
)

Test MC probabilities: (20, 992, 7)
Test labels: (992,)


In [116]:
# Phase 9 correction: Verify labels and stochastic behavior

seed42_val_labels = np.load(
    output_dirs["logits_dir"]
    / "val_labels_seed42.npy"
)

seed42_test_labels = np.load(
    output_dirs["logits_dir"]
    / "test_labels_seed42.npy"
)

val_labels_match = np.array_equal(
    seed42_val_labels,
    val_mc_labels,
)

test_labels_match = np.array_equal(
    seed42_test_labels,
    test_mc_labels,
)

val_probability_sums_valid = np.allclose(
    val_mc_probabilities.sum(axis=2),
    1.0,
)

test_probability_sums_valid = np.allclose(
    test_mc_probabilities.sum(axis=2),
    1.0,
)

test_variance = np.var(
    test_mc_probabilities,
    axis=0,
)

maximum_pass_difference = np.max(
    np.abs(
        test_mc_probabilities[0]
        - test_mc_probabilities[1]
    )
)

all_passes_identical = np.allclose(
    test_mc_probabilities,
    test_mc_probabilities[0][None, :, :],
)

print("Validation labels match:", val_labels_match)
print("Test labels match:", test_labels_match)
print(
    "Validation probability sums valid:",
    val_probability_sums_valid,
)
print(
    "Test probability sums valid:",
    test_probability_sums_valid,
)
print(
    "Maximum pass difference:",
    maximum_pass_difference,
)
print(
    "Mean probability variance:",
    test_variance.mean(),
)
print(
    "Maximum probability variance:",
    test_variance.max(),
)
print(
    "All passes identical:",
    all_passes_identical,
)

assert val_labels_match
assert test_labels_match
assert val_probability_sums_valid
assert test_probability_sums_valid
assert maximum_pass_difference > 0
assert test_variance.mean() > 0
assert not all_passes_identical

print(
    "Corrected MC Dropout stochasticity verification passed."
)

Validation labels match: True
Test labels match: True
Validation probability sums valid: True
Test probability sums valid: True
Maximum pass difference: 0.7592361
Mean probability variance: 0.0026828628
Maximum probability variance: 0.09744145
All passes identical: False
Corrected MC Dropout stochasticity verification passed.


In [117]:
# Phase 9 correction: Save stochastic MC Dropout probabilities

val_mc_members_path = (
    output_dirs["logits_dir"]
    / "val_probs_mc_dropout_members.npy"
)

test_mc_members_path = (
    output_dirs["logits_dir"]
    / "test_probs_mc_dropout_members.npy"
)

np.save(
    val_mc_members_path,
    val_mc_probabilities,
)

np.save(
    test_mc_members_path,
    test_mc_probabilities,
)

print("Saved:", val_mc_members_path)
print("Saved:", test_mc_members_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/logits/val_probs_mc_dropout_members.npy
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/logits/test_probs_mc_dropout_members.npy


In [118]:
# Phase 9 correction: Compute MC Dropout mean probabilities

val_mc_mean_probs = ensemble_mean_probabilities(
    val_mc_probabilities
)

test_mc_mean_probs = ensemble_mean_probabilities(
    test_mc_probabilities
)

val_mc_mean_path = (
    output_dirs["logits_dir"]
    / "val_probs_mc_dropout_uncalibrated.npy"
)

test_mc_mean_path = (
    output_dirs["logits_dir"]
    / "test_probs_mc_dropout_uncalibrated.npy"
)

np.save(
    val_mc_mean_path,
    val_mc_mean_probs,
)

np.save(
    test_mc_mean_path,
    test_mc_mean_probs,
)

print("Validation MC mean:", val_mc_mean_probs.shape)
print("Test MC mean:", test_mc_mean_probs.shape)

print(
    "Validation mean sums valid:",
    np.allclose(
        val_mc_mean_probs.sum(axis=1),
        1.0,
    ),
)

print(
    "Test mean sums valid:",
    np.allclose(
        test_mc_mean_probs.sum(axis=1),
        1.0,
    ),
)

Validation MC mean: (1010, 7)
Test MC mean: (992, 7)
Validation mean sums valid: True
Test mean sums valid: True


In [119]:
# Phase 9 correction: Evaluate uncalibrated MC Dropout

mc_classification_uncalibrated = (
    classification_metrics_from_probabilities(
        probs=test_mc_mean_probs,
        labels=test_mc_labels,
        classes=classes,
    )
)

mc_calibration_uncalibrated = (
    calibration_metrics_from_probabilities(
        probs=test_mc_mean_probs,
        labels=test_mc_labels,
        classes=classes,
        n_bins=15,
    )
)

print("Uncalibrated MC Dropout classification:")
display(
    pd.DataFrame(
        [mc_classification_uncalibrated]
    )
)

print("Uncalibrated MC Dropout calibration:")
display(
    pd.DataFrame(
        [mc_calibration_uncalibrated]
    )
)

Uncalibrated MC Dropout classification:


,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall
0,0.813508,0.679339,0.75725,0.713135,0.675676


Uncalibrated MC Dropout calibration:


,accuracy,nll,brier_score,ece
0,0.813508,0.574104,0.276393,0.070518


In [120]:
# Phase 9 correction: Fit MC Dropout Temperature Scaling

mc_temperature = (
    fit_temperature_on_log_probabilities(
        probabilities=val_mc_mean_probs,
        labels=val_mc_labels,
        max_iter=1000,
        lr=0.01,
        initial_temperature=1.0,
        device=(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        ),
    )
)

mc_temperature_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_temperature.csv"
)

save_temperature(
    mc_temperature,
    mc_temperature_path,
)

print(
    "Learned MC Dropout temperature:",
    mc_temperature,
)

print("Saved:", mc_temperature_path)

Learned MC Dropout temperature: 1.599391222000122
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/mc_dropout_temperature.csv


In [121]:
# Phase 9 correction: Apply Temperature Scaling to MC Dropout

val_mc_probs_calibrated = (
    apply_temperature_to_probabilities(
        probabilities=val_mc_mean_probs,
        temperature=mc_temperature,
    )
)

test_mc_probs_calibrated = (
    apply_temperature_to_probabilities(
        probabilities=test_mc_mean_probs,
        temperature=mc_temperature,
    )
)

val_mc_calibrated_path = (
    output_dirs["logits_dir"]
    / "val_probs_mc_dropout_calibrated.npy"
)

test_mc_calibrated_path = (
    output_dirs["logits_dir"]
    / "test_probs_mc_dropout_calibrated.npy"
)

np.save(
    val_mc_calibrated_path,
    val_mc_probs_calibrated,
)

np.save(
    test_mc_calibrated_path,
    test_mc_probs_calibrated,
)

print(
    "Validation calibrated:",
    val_mc_probs_calibrated.shape,
)

print(
    "Test calibrated:",
    test_mc_probs_calibrated.shape,
)

print(
    "Calibrated test sums valid:",
    np.allclose(
        test_mc_probs_calibrated.sum(axis=1),
        1.0,
    ),
)

Validation calibrated: (1010, 7)
Test calibrated: (992, 7)
Calibrated test sums valid: True


In [122]:
# Phase 9 correction: Evaluate calibrated MC Dropout

mc_classification_calibrated = (
    classification_metrics_from_probabilities(
        probs=test_mc_probs_calibrated,
        labels=test_mc_labels,
        classes=classes,
    )
)

mc_calibration_calibrated = (
    calibration_metrics_from_probabilities(
        probs=test_mc_probs_calibrated,
        labels=test_mc_labels,
        classes=classes,
        n_bins=15,
    )
)

mc_metrics_df = pd.DataFrame([
    {
        "model": "mc_dropout_uncalibrated",
        "temperature": np.nan,
        **mc_classification_uncalibrated,
        "nll": mc_calibration_uncalibrated["nll"],
        "brier_score":
            mc_calibration_uncalibrated[
                "brier_score"
            ],
        "ece": mc_calibration_uncalibrated["ece"],
    },
    {
        "model": "mc_dropout_temperature_scaled",
        "temperature": mc_temperature,
        **mc_classification_calibrated,
        "nll": mc_calibration_calibrated["nll"],
        "brier_score":
            mc_calibration_calibrated[
                "brier_score"
            ],
        "ece": mc_calibration_calibrated["ece"],
    },
])

mc_metrics_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_metrics.csv"
)

mc_metrics_df.to_csv(
    mc_metrics_path,
    index=False,
)

display(mc_metrics_df)
print("Saved:", mc_metrics_path)

,model,temperature,accuracy,macro_precision,macro_recall,macro_f1,melanoma_recall,nll,brier_score,ece
0,mc_dropout_uncalibrated,NaN,0.813508,0.679339,0.75725,0.713135,0.675676,0.574104,0.276393,0.070518
1,mc_dropout_temperature_scaled,1.599391,0.813508,0.679339,0.75725,0.713135,0.675676,0.507717,0.266311,0.031193


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/mc_dropout_metrics.csv


In [123]:
# Phase 9 correction: Generate MC Dropout uncertainty scores

metadata_df = test_mc_metadata[
    ["image_id", "lesion_id"]
].copy()

mc_scores_df = (
    mc_dropout_uncertainty_scores_dataframe(
        stochastic_probabilities=(
            test_mc_probabilities
        ),
        labels=test_mc_labels,
        classes=classes,
        metadata_df=metadata_df,
    )
)

mc_scores_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_uncertainty_scores.csv"
)

mc_scores_df.to_csv(
    mc_scores_path,
    index=False,
)

display(mc_scores_df.head())

display(
    mc_scores_df[
        [
            "confidence",
            "confidence_uncertainty",
            "predictive_entropy",
            "expected_entropy",
            "mutual_information",
            "probability_variance",
            "correct",
        ]
    ].describe()
)

print("Saved:", mc_scores_path)

,image_id,lesion_id,true_label,true_class,predicted_label,predicted_class,confidence,confidence_uncertainty,predictive_entropy,expected_entropy,mutual_information,probability_variance,correct
0,ISIC_0027419,HAM_0000118,2,bkl,2,bkl,0.964996,0.035004,0.193591,0.175953,0.017638,0.000294,True
1,ISIC_0025030,HAM_0000118,2,bkl,2,bkl,0.979816,0.020184,0.112206,0.105213,0.006993,0.000062,True
2,ISIC_0029176,HAM_0002761,2,bkl,2,bkl,0.924554,0.075446,0.327359,0.290426,0.036933,0.001240,True
3,ISIC_0029068,HAM_0002761,2,bkl,1,bcc,0.541372,0.458628,1.228715,1.115856,0.112859,0.008496,False
4,ISIC_0032343,HAM_0006071,2,bkl,2,bkl,0.789731,0.210269,0.547318,0.462567,0.084752,0.007701,True


,confidence,confidence_uncertainty,predictive_entropy,expected_entropy,mutual_information,probability_variance
count,992.000000,992.000000,9.920000e+02,9.920000e+02,992.000000,9.920000e+02
mean,0.877961,0.122039,3.070635e-01,2.740216e-01,0.033042,2.682863e-03
std,0.173085,0.173085,3.790503e-01,3.405468e-01,0.045085,4.392826e-03
min,0.284815,0.000000,1.657861e-10,1.657861e-10,0.000000,4.068978e-43
25%,0.784060,0.000101,1.085830e-03,9.598162e-04,0.000084,6.057117e-09
50%,0.984566,0.015434,9.122394e-02,8.023944e-02,0.007443,6.629495e-05
75%,0.999899,0.215940,6.071338e-01,5.288928e-01,0.060777,4.347206e-03
max,1.000000,0.715185,1.686876e+00,1.590718e+00,0.326634,2.672497e-02


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/mc_dropout_uncertainty_scores.csv


In [124]:
# Phase 9 correction: Compute MC Dropout error-detection AUROC

mc_error_detection_df = (
    mc_dropout_error_detection_table(
        mc_scores_df
    )
)

mc_error_detection_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_error_detection.csv"
)

mc_error_detection_df.to_csv(
    mc_error_detection_path,
    index=False,
)

display(mc_error_detection_df)
print("Saved:", mc_error_detection_path)

,score,error_detection_auroc
0,confidence_uncertainty,0.836123
1,predictive_entropy,0.838019
2,mutual_information,0.821836
3,probability_variance,0.822064


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/mc_dropout_error_detection.csv


In [125]:
# Phase 9 correction: Compute MC Dropout referral results

mc_referral_frames = []

mc_uncertainty_methods = {
    "mc_dropout_confidence_uncertainty":
        mc_scores_df[
            "confidence_uncertainty"
        ].to_numpy(),

    "mc_dropout_predictive_entropy":
        mc_scores_df[
            "predictive_entropy"
        ].to_numpy(),

    "mc_dropout_mutual_information":
        mc_scores_df[
            "mutual_information"
        ].to_numpy(),

    "mc_dropout_probability_variance":
        mc_scores_df[
            "probability_variance"
        ].to_numpy(),
}

for method_name, uncertainty_values in (
    mc_uncertainty_methods.items()
):
    method_df = referral_results_at_coverages(
        probs=test_mc_probs_calibrated,
        labels=test_mc_labels,
        uncertainty_scores=uncertainty_values,
        coverages=coverages,
        classes=classes,
        method_name=method_name,
    )

    mc_referral_frames.append(method_df)

mc_referral_df = pd.concat(
    mc_referral_frames,
    ignore_index=True,
)

mc_referral_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_referral_results.csv"
)

mc_referral_df.to_csv(
    mc_referral_path,
    index=False,
)

display(mc_referral_df)
print("Saved:", mc_referral_path)

,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1
0,mc_dropout_confidence_uncertainty,1.000000,0.000000,992,0,0.813508,0.186492,0.713135
1,mc_dropout_confidence_uncertainty,0.950605,0.049395,943,49,0.832450,0.167550,0.735771
2,mc_dropout_confidence_uncertainty,0.900202,0.099798,893,99,0.847704,0.152296,0.748380
3,mc_dropout_confidence_uncertainty,0.850806,0.149194,844,148,0.869668,0.130332,0.774292
4,mc_dropout_confidence_uncertainty,0.800403,0.199597,794,198,0.886650,0.113350,0.798283
5,mc_dropout_confidence_uncertainty,0.700605,0.299395,695,297,0.926619,0.073381,0.850713
6,mc_dropout_confidence_uncertainty,0.600806,0.399194,596,396,0.944631,0.055369,0.862772
7,mc_dropout_predictive_entropy,1.000000,0.000000,992,0,0.813508,0.186492,0.713135
8,mc_dropout_predictive_entropy,0.950605,0.049395,943,49,0.835631,0.164369,0.742664
9,mc_dropout_predictive_entropy,0.900202,0.099798,893,99,0.851064,0.148936,0.763112


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/mc_dropout_referral_results.csv


In [126]:
# Phase 9 correction: Select referral threshold using validation data

val_mc_confidence_uncertainty = (
    1.0
    - np.max(
        val_mc_probs_calibrated,
        axis=1,
    )
)

test_mc_confidence_uncertainty = (
    1.0
    - np.max(
        test_mc_probs_calibrated,
        axis=1,
    )
)

target_coverage = 0.90

mc_selected_threshold = (
    select_uncertainty_threshold_for_coverage(
        uncertainty_scores=(
            val_mc_confidence_uncertainty
        ),
        target_coverage=target_coverage,
    )
)

mc_selected_result = (
    evaluate_referral_threshold(
        probs=test_mc_probs_calibrated,
        labels=test_mc_labels,
        uncertainty_scores=(
            test_mc_confidence_uncertainty
        ),
        threshold=mc_selected_threshold,
        classes=classes,
        method_name=(
            "validation_selected_"
            "mc_dropout_confidence_uncertainty"
        ),
    )
)

mc_threshold_df = pd.DataFrame([{
    "method":
        "mc_dropout_confidence_uncertainty",
    "target_validation_coverage":
        target_coverage,
    "selected_threshold":
        mc_selected_threshold,
}])

mc_selected_result_df = pd.DataFrame([
    mc_selected_result
])

mc_threshold_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_selected_referral_threshold.csv"
)

mc_selected_result_path = (
    output_dirs["tables_dir"]
    / "mc_dropout_selected_referral_test_result.csv"
)

mc_threshold_df.to_csv(
    mc_threshold_path,
    index=False,
)

mc_selected_result_df.to_csv(
    mc_selected_result_path,
    index=False,
)

display(mc_threshold_df)
display(mc_selected_result_df)

,method,target_validation_coverage,selected_threshold
0,mc_dropout_confidence_uncertainty,0.9,0.505763


,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1,threshold
0,validation_selected_mc_dropout_confidence_unce...,0.893145,0.106855,886,106,0.852144,0.147856,0.762772,0.505763


In [127]:
# Phase 9 correction: Save predictions and referral decisions

mc_predictions_df = (
    prediction_dataframe_from_probabilities(
        probs=test_mc_probs_calibrated,
        labels=test_mc_labels,
        metadata_df=metadata_df,
        classes=classes,
    )
)

mc_predictions_df[
    "confidence_uncertainty"
] = test_mc_confidence_uncertainty

mc_decision_df = add_referral_decision(
    scores_df=mc_predictions_df,
    uncertainty_column="confidence_uncertainty",
    threshold=mc_selected_threshold,
)

mc_predictions_path = (
    output_dirs["predictions_dir"]
    / "test_predictions_mc_dropout_calibrated.csv"
)

mc_decision_path = (
    output_dirs["predictions_dir"]
    / "test_referral_decisions_mc_dropout.csv"
)

mc_predictions_df.to_csv(
    mc_predictions_path,
    index=False,
)

mc_decision_df.to_csv(
    mc_decision_path,
    index=False,
)

display(
    mc_decision_df["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="count")
)

print("Saved:", mc_predictions_path)
print("Saved:", mc_decision_path)

,decision,count
0,accept,886
1,refer,106


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_predictions_mc_dropout_calibrated.csv
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/predictions/test_referral_decisions_mc_dropout.csv


In [128]:
# Phase 9 correction: Create MC Dropout reliability diagram

mc_uncalibrated_curve_df = (
    reliability_curve_dataframe(
        probs=test_mc_mean_probs,
        labels=test_mc_labels,
        n_bins=15,
    )
)

mc_calibrated_curve_df = (
    reliability_curve_dataframe(
        probs=test_mc_probs_calibrated,
        labels=test_mc_labels,
        n_bins=15,
    )
)

mc_reliability_plot_path = (
    plot_reliability_diagram(
        uncalibrated_curve_df=(
            mc_uncalibrated_curve_df
        ),
        calibrated_curve_df=(
            mc_calibrated_curve_df
        ),
        output_path=(
            output_dirs["figures_dir"]
            / "mc_dropout_reliability_diagram.png"
        ),
    )
)

print("Saved:", mc_reliability_plot_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/mc_dropout_reliability_diagram.png


In [129]:
# Phase 9 correction: Create MC Dropout uncertainty plot

mc_uncertainty_plot_path = (
    plot_uncertainty_correct_vs_incorrect(
        scores_df=mc_scores_df,
        score_column="mutual_information",
        output_path=(
            output_dirs["figures_dir"]
            / "mc_dropout_uncertainty_correct_vs_incorrect.png"
        ),
    )
)

print("Saved:", mc_uncertainty_plot_path)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/mc_dropout_uncertainty_correct_vs_incorrect.png


In [130]:
# Phase 9 correction: Create final risk-coverage comparison

baseline_referral_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "baseline_referral_results_seed42.csv"
)

ensemble_referral_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "deep_ensemble_referral_results.csv"
)

combined_risk_coverage_df = pd.concat(
    [
        baseline_referral_df[
            baseline_referral_df["method"]
            == "calibrated_confidence_uncertainty"
        ],
        ensemble_referral_df[
            ensemble_referral_df["method"]
            == "ensemble_confidence_uncertainty"
        ],
        mc_referral_df[
            mc_referral_df["method"]
            == "mc_dropout_confidence_uncertainty"
        ],
    ],
    ignore_index=True,
)

combined_risk_coverage_path = (
    output_dirs["tables_dir"]
    / "final_risk_coverage_comparison.csv"
)

combined_risk_coverage_df.to_csv(
    combined_risk_coverage_path,
    index=False,
)

final_risk_coverage_plot = (
    plot_risk_coverage_curve(
        referral_results_df=(
            combined_risk_coverage_df
        ),
        output_path=(
            output_dirs["figures_dir"]
            / "final_risk_coverage_comparison.png"
        ),
    )
)

display(combined_risk_coverage_df)

print("Saved:", combined_risk_coverage_path)
print("Saved:", final_risk_coverage_plot)

,method,coverage,rejection_rate,accepted_count,rejected_count,selective_accuracy,selective_risk,macro_f1
0,calibrated_confidence_uncertainty,1.000000,0.000000,992,0,0.812500,0.187500,0.714404
1,calibrated_confidence_uncertainty,0.950605,0.049395,943,49,0.834571,0.165429,0.736751
2,calibrated_confidence_uncertainty,0.900202,0.099798,893,99,0.849944,0.150056,0.757617
3,calibrated_confidence_uncertainty,0.850806,0.149194,844,148,0.864929,0.135071,0.777277
4,calibrated_confidence_uncertainty,0.800403,0.199597,794,198,0.886650,0.113350,0.807588
5,calibrated_confidence_uncertainty,0.700605,0.299395,695,297,0.923741,0.076259,0.850408
6,calibrated_confidence_uncertainty,0.600806,0.399194,596,396,0.953020,0.046980,0.878084
7,ensemble_confidence_uncertainty,1.000000,0.000000,992,0,0.811492,0.188508,0.730762
8,ensemble_confidence_uncertainty,0.950605,0.049395,943,49,0.835631,0.164369,0.779449
9,ensemble_confidence_uncertainty,0.900202,0.099798,893,99,0.853303,0.146697,0.808020


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/final_risk_coverage_comparison.csv
Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/final_risk_coverage_comparison.png


In [131]:
# Phase 9 correction: Build final model comparison table

phase8_comparison_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "baseline_vs_deep_ensemble_comparison.csv"
)

final_model_comparison_df = pd.concat(
    [
        phase8_comparison_df,
        mc_metrics_df,
    ],
    ignore_index=True,
    sort=False,
)

final_model_comparison_path = (
    output_dirs["tables_dir"]
    / "final_model_comparison.csv"
)

final_model_comparison_df.to_csv(
    final_model_comparison_path,
    index=False,
)

display(final_model_comparison_df)
print("Saved:", final_model_comparison_path)

,model,accuracy,macro_f1,melanoma_recall,nll,brier_score,ece,temperature,macro_precision,macro_recall
0,single_seed42_uncalibrated,0.812500,0.714404,0.675676,0.600292,0.280098,0.082272,NaN,NaN,NaN
1,single_seed42_temperature_scaled,0.812500,0.714404,0.675676,0.505970,0.264806,0.030612,NaN,NaN,NaN
2,deep_ensemble_uncalibrated,0.811492,0.730762,0.774775,0.476446,0.257427,0.016339,NaN,NaN,NaN
3,deep_ensemble_temperature_scaled,0.811492,0.730762,0.774775,0.475816,0.257357,0.020517,NaN,NaN,NaN
4,mc_dropout_uncalibrated,0.813508,0.713135,0.675676,0.574104,0.276393,0.070518,NaN,0.679339,0.75725
5,mc_dropout_temperature_scaled,0.813508,0.713135,0.675676,0.507717,0.266311,0.031193,1.599391,0.679339,0.75725


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/final_model_comparison.csv


In [132]:
# Phase 9 correction: Plot final classification comparison

final_classification_plot = (
    plot_model_comparison(
        comparison_df=final_model_comparison_df,
        metric_columns=[
            "accuracy",
            "macro_f1",
            "melanoma_recall",
        ],
        output_path=(
            output_dirs["figures_dir"]
            / "final_classification_comparison.png"
        ),
    )
)

print("Saved:", final_classification_plot)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/final_classification_comparison.png


In [133]:
# Phase 9 correction: Plot final calibration comparison

final_calibration_plot = (
    plot_model_comparison(
        comparison_df=final_model_comparison_df,
        metric_columns=[
            "nll",
            "brier_score",
            "ece",
        ],
        output_path=(
            output_dirs["figures_dir"]
            / "final_calibration_comparison.png"
        ),
    )
)

print("Saved:", final_calibration_plot)

Saved: /kaggle/working/skin-lesion-uncertainty/outputs/figures/final_calibration_comparison.png


In [134]:
# Phase 9 correction: Build final uncertainty comparison

baseline_error_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "baseline_uncertainty_error_detection_seed42.csv"
)

ensemble_error_df = pd.read_csv(
    output_dirs["tables_dir"]
    / "deep_ensemble_error_detection.csv"
)

def select_auroc(df, score_name):
    row = df[df["score"] == score_name]

    if row.empty:
        return np.nan

    return float(
        row.iloc[0]["error_detection_auroc"]
    )

uncertainty_comparison_df = pd.DataFrame([
    {
        "method": "single_model",
        "confidence_uncertainty_auroc":
            select_auroc(
                baseline_error_df,
                "confidence_uncertainty",
            ),
        "predictive_entropy_auroc":
            select_auroc(
                baseline_error_df,
                "predictive_entropy",
            ),
    },
    {
        "method": "deep_ensemble",
        "confidence_uncertainty_auroc":
            select_auroc(
                ensemble_error_df,
                "confidence_uncertainty",
            ),
        "predictive_entropy_auroc":
            select_auroc(
                ensemble_error_df,
                "predictive_entropy",
            ),
        "mutual_information_auroc":
            select_auroc(
                ensemble_error_df,
                "mutual_information",
            ),
        "probability_variance_auroc":
            select_auroc(
                ensemble_error_df,
                "probability_variance",
            ),
    },
    {
        "method": "mc_dropout",
        "confidence_uncertainty_auroc":
            select_auroc(
                mc_error_detection_df,
                "confidence_uncertainty",
            ),
        "predictive_entropy_auroc":
            select_auroc(
                mc_error_detection_df,
                "predictive_entropy",
            ),
        "mutual_information_auroc":
            select_auroc(
                mc_error_detection_df,
                "mutual_information",
            ),
        "probability_variance_auroc":
            select_auroc(
                mc_error_detection_df,
                "probability_variance",
            ),
    },
])

uncertainty_comparison_path = (
    output_dirs["tables_dir"]
    / "final_uncertainty_comparison.csv"
)

uncertainty_comparison_df.to_csv(
    uncertainty_comparison_path,
    index=False,
)

display(uncertainty_comparison_df)
print("Saved:", uncertainty_comparison_path)

,method,confidence_uncertainty_auroc,predictive_entropy_auroc,mutual_information_auroc,probability_variance_auroc
0,single_model,0.843412,0.838049,NaN,NaN
1,deep_ensemble,0.867792,0.867752,0.796964,0.804092
2,mc_dropout,0.836123,0.838019,0.821836,0.822064


Saved: /kaggle/working/skin-lesion-uncertainty/outputs/tables/final_uncertainty_comparison.csv


In [135]:
# Phase 9 correction: Verify all corrected Phase 9 files

phase9_expected_files = [
    output_dirs["logits_dir"]
    / "val_probs_mc_dropout_members.npy",

    output_dirs["logits_dir"]
    / "test_probs_mc_dropout_members.npy",

    output_dirs["logits_dir"]
    / "val_probs_mc_dropout_uncalibrated.npy",

    output_dirs["logits_dir"]
    / "test_probs_mc_dropout_uncalibrated.npy",

    output_dirs["logits_dir"]
    / "val_probs_mc_dropout_calibrated.npy",

    output_dirs["logits_dir"]
    / "test_probs_mc_dropout_calibrated.npy",

    output_dirs["tables_dir"]
    / "mc_dropout_temperature.csv",

    output_dirs["tables_dir"]
    / "mc_dropout_metrics.csv",

    output_dirs["tables_dir"]
    / "mc_dropout_uncertainty_scores.csv",

    output_dirs["tables_dir"]
    / "mc_dropout_error_detection.csv",

    output_dirs["tables_dir"]
    / "mc_dropout_referral_results.csv",

    output_dirs["tables_dir"]
    / "mc_dropout_selected_referral_threshold.csv",

    output_dirs["tables_dir"]
    / "mc_dropout_selected_referral_test_result.csv",

    output_dirs["tables_dir"]
    / "final_risk_coverage_comparison.csv",

    output_dirs["tables_dir"]
    / "final_model_comparison.csv",

    output_dirs["tables_dir"]
    / "final_uncertainty_comparison.csv",

    output_dirs["predictions_dir"]
    / "test_predictions_mc_dropout_calibrated.csv",

    output_dirs["predictions_dir"]
    / "test_referral_decisions_mc_dropout.csv",

    output_dirs["figures_dir"]
    / "mc_dropout_reliability_diagram.png",

    output_dirs["figures_dir"]
    / "mc_dropout_uncertainty_correct_vs_incorrect.png",

    output_dirs["figures_dir"]
    / "final_risk_coverage_comparison.png",

    output_dirs["figures_dir"]
    / "final_classification_comparison.png",

    output_dirs["figures_dir"]
    / "final_calibration_comparison.png",
]

verification_rows = []

for path in phase9_expected_files:
    exists = path.exists()
    size_bytes = (
        path.stat().st_size
        if exists
        else 0
    )

    verification_rows.append({
        "file": str(
            path.relative_to(REPO_DIR)
        ),
        "exists": exists,
        "non_empty": size_bytes > 0,
        "size_bytes": size_bytes,
    })

phase9_verification_df = pd.DataFrame(
    verification_rows
)

display(phase9_verification_df)

phase9_all_ok = (
    phase9_verification_df["exists"].all()
    and phase9_verification_df[
        "non_empty"
    ].all()
)

print(
    "\nAll corrected Phase 9 files exist "
    "and are non-empty:",
    phase9_all_ok,
)

,file,exists,non_empty,size_bytes
0,outputs/logits/val_probs_mc_dropout_members.npy,True,True,565728
1,outputs/logits/test_probs_mc_dropout_members.npy,True,True,555648
2,outputs/logits/val_probs_mc_dropout_uncalibrat...,True,True,56688
3,outputs/logits/test_probs_mc_dropout_uncalibra...,True,True,55680
4,outputs/logits/val_probs_mc_dropout_calibrated...,True,True,56688
5,outputs/logits/test_probs_mc_dropout_calibrate...,True,True,55680
6,outputs/tables/mc_dropout_temperature.csv,True,True,39
7,outputs/tables/mc_dropout_metrics.csv,True,True,479
8,outputs/tables/mc_dropout_uncertainty_scores.csv,True,True,160354
9,outputs/tables/mc_dropout_error_detection.csv,True,True,186



All corrected Phase 9 files exist and are non-empty: True


In [136]:
# Phase 10: Back up all completed outputs

from pathlib import Path
from datetime import datetime
import shutil

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_zip = shutil.make_archive(
    base_name=str(
        Path("/kaggle/working")
        / f"final_outputs_backup_{timestamp}"
    ),
    format="zip",
    root_dir=REPO_DIR,
    base_dir="outputs",
)

print("Final backup created:", backup_zip)
print("Backup size:", Path(backup_zip).stat().st_size, "bytes")

Final backup created: /kaggle/working/final_outputs_backup_20260725_142536.zip
Backup size: 139126346 bytes


In [143]:
# Phase 10: Verify final analysis artifacts before packaging

from pathlib import Path
import pandas as pd

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
OUTPUTS_DIR = REPO_DIR / "outputs"

critical_files = [
    # Configuration and project information
    REPO_DIR / "README.md",
    REPO_DIR / "configs/experiment.yaml",

    # Dataset split information
    OUTPUTS_DIR / "tables/train_split.csv",
    OUTPUTS_DIR / "tables/val_split.csv",
    OUTPUTS_DIR / "tables/test_split.csv",
    OUTPUTS_DIR / "tables/class_distribution_by_split.csv",

    # Baseline
    OUTPUTS_DIR / "tables/baseline_seed42_metrics.csv",
    OUTPUTS_DIR / "tables/training_history_seed42.csv",
    OUTPUTS_DIR / "tables/calibration_metrics_seed42.csv",
    OUTPUTS_DIR / "tables/baseline_uncertainty_error_detection_seed42.csv",
    OUTPUTS_DIR / "tables/baseline_referral_results_seed42.csv",

    # Ensemble members
    OUTPUTS_DIR / "tables/ensemble_individual_member_metrics.csv",

    # Deep Ensemble
    OUTPUTS_DIR / "tables/deep_ensemble_metrics.csv",
    OUTPUTS_DIR / "tables/deep_ensemble_error_detection.csv",
    OUTPUTS_DIR / "tables/deep_ensemble_referral_results.csv",
    OUTPUTS_DIR / "tables/deep_ensemble_selected_referral_test_result.csv",

    # MC Dropout
    OUTPUTS_DIR / "tables/mc_dropout_metrics.csv",
    OUTPUTS_DIR / "tables/mc_dropout_error_detection.csv",
    OUTPUTS_DIR / "tables/mc_dropout_referral_results.csv",
    OUTPUTS_DIR / "tables/mc_dropout_selected_referral_test_result.csv",

    # Final comparisons
    OUTPUTS_DIR / "tables/final_model_comparison.csv",
    OUTPUTS_DIR / "tables/final_uncertainty_comparison.csv",
    OUTPUTS_DIR / "tables/final_risk_coverage_comparison.csv",

    # Important final figures
    OUTPUTS_DIR / "figures/final_classification_comparison.png",
    OUTPUTS_DIR / "figures/final_calibration_comparison.png",
    OUTPUTS_DIR / "figures/final_risk_coverage_comparison.png",
    OUTPUTS_DIR / "figures/deep_ensemble_reliability_diagram.png",
]

verification_rows = []

for path in critical_files:
    exists = path.exists()
    size_bytes = path.stat().st_size if exists else 0

    verification_rows.append({
        "file": str(path.relative_to(REPO_DIR)),
        "exists": exists,
        "non_empty": size_bytes > 0,
        "size_bytes": size_bytes,
    })

critical_verification_df = pd.DataFrame(verification_rows)
display(critical_verification_df)

critical_ok = (
    critical_verification_df["exists"].all()
    and critical_verification_df["non_empty"].all()
)

print("\nAll critical analysis files are ready:", critical_ok)

,file,exists,non_empty,size_bytes
0,README.md,True,True,6067
1,configs/experiment.yaml,True,True,2588
2,outputs/tables/train_split.csv,True,True,1276011
3,outputs/tables/val_split.csv,True,True,159032
4,outputs/tables/test_split.csv,True,True,156920
5,outputs/tables/class_distribution_by_split.csv,True,True,125
6,outputs/tables/baseline_seed42_metrics.csv,True,True,145
7,outputs/tables/training_history_seed42.csv,True,True,2209
8,outputs/tables/calibration_metrics_seed42.csv,True,True,244
9,outputs/tables/baseline_uncertainty_error_dete...,True,True,108



All critical analysis files are ready: True


In [144]:
# Phase 10: Build analysis-ready project package

from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import sys
import platform
import json
import hashlib
import pandas as pd
import importlib.metadata as importlib_metadata

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
PACKAGE_ROOT = Path("/kaggle/working/skin_lesion_analysis_package")

# Remove an older staging folder only, not project outputs.
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)

PACKAGE_ROOT.mkdir(parents=True)

# Create package directories.
metadata_dir = PACKAGE_ROOT / "project_metadata"
source_dir = PACKAGE_ROOT / "source_snapshot"
environment_dir = PACKAGE_ROOT / "environment"
package_outputs_dir = PACKAGE_ROOT / "outputs"
manifest_dir = PACKAGE_ROOT / "manifest"

for directory in [
    metadata_dir,
    source_dir,
    environment_dir,
    package_outputs_dir,
    manifest_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Copy project metadata.
metadata_files = [
    "README.md",
    ".gitignore",
    "requirements.txt",
    "requirements-local.txt",
]

for filename in metadata_files:
    source_path = REPO_DIR / filename
    if source_path.exists():
        shutil.copy2(source_path, metadata_dir / filename)

# Copy configuration.
config_source = REPO_DIR / "configs"
if config_source.exists():
    shutil.copytree(
        config_source,
        metadata_dir / "configs",
        dirs_exist_ok=True,
    )

# Copy source and tests.
for folder_name in ["src", "tests"]:
    source_path = REPO_DIR / folder_name
    if source_path.exists():
        shutil.copytree(
            source_path,
            source_dir / folder_name,
            dirs_exist_ok=True,
            ignore=shutil.ignore_patterns(
                "__pycache__",
                "*.pyc",
                ".pytest_cache",
            ),
        )

# Copy analysis outputs, excluding checkpoints.
for output_folder in [
    "tables",
    "figures",
    "predictions",
    "logits",
]:
    source_path = REPO_DIR / "outputs" / output_folder

    if source_path.exists():
        shutil.copytree(
            source_path,
            package_outputs_dir / output_folder,
            dirs_exist_ok=True,
            ignore=shutil.ignore_patterns(
                "*.zip",
                "__pycache__",
            ),
        )

# Save Git commit and Git history.
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

git_log = subprocess.run(
    ["git", "log", "--oneline", "--decorate", "-20"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

(environment_dir / "git_commit.txt").write_text(
    git_commit + "\n",
    encoding="utf-8",
)

(environment_dir / "git_log.txt").write_text(
    git_log + "\n",
    encoding="utf-8",
)

# Save environment information.
python_environment = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
}

try:
    import torch

    python_environment.update({
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
    })
except Exception as exc:
    python_environment["torch_error"] = repr(exc)

(environment_dir / "python_environment.txt").write_text(
    json.dumps(
        python_environment,
        indent=2,
    ),
    encoding="utf-8",
)

# Save important package versions.
important_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "torch",
    "torchvision",
    "timm",
    "torchmetrics",
    "PyYAML",
    "Pillow",
]

package_versions = {}

for package_name in important_packages:
    try:
        package_versions[package_name] = (
            importlib_metadata.version(package_name)
        )
    except importlib_metadata.PackageNotFoundError:
        package_versions[package_name] = "not-installed"

(environment_dir / "package_versions.txt").write_text(
    json.dumps(
        package_versions,
        indent=2,
    ),
    encoding="utf-8",
)

# Create a package summary.
package_summary = {
    "package_name": "skin_lesion_analysis_package",
    "created_at": datetime.now().isoformat(),
    "git_commit": git_commit,
    "project_title": (
        "Uncertainty-Aware Skin Lesion Classification "
        "with Human Referral"
    ),
    "dataset": "HAM10000",
    "classes": [
        "akiec",
        "bcc",
        "bkl",
        "df",
        "mel",
        "nv",
        "vasc",
    ],
    "ensemble_seeds": [42, 123, 999],
    "mc_dropout_passes": 20,
    "included_output_folders": [
        "tables",
        "figures",
        "predictions",
        "logits",
    ],
    "checkpoints_included": False,
    "purpose": (
        "Detailed analysis, report preparation, result validation, "
        "figure selection, and final submission planning."
    ),
}

(manifest_dir / "package_summary.json").write_text(
    json.dumps(
        package_summary,
        indent=2,
    ),
    encoding="utf-8",
)

# Build inventory and SHA256 checksums.
inventory_rows = []
checksum_lines = []

for path in sorted(PACKAGE_ROOT.rglob("*")):
    if not path.is_file():
        continue

    relative_path = path.relative_to(PACKAGE_ROOT)
    file_size = path.stat().st_size

    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    checksum = sha256.hexdigest()

    inventory_rows.append({
        "relative_path": str(relative_path),
        "extension": path.suffix.lower(),
        "size_bytes": file_size,
        "size_mb": round(file_size / 1024**2, 4),
        "sha256": checksum,
    })

    checksum_lines.append(
        f"{checksum}  {relative_path}"
    )

inventory_df = pd.DataFrame(inventory_rows)

inventory_df.to_csv(
    manifest_dir / "file_inventory.csv",
    index=False,
)

(manifest_dir / "checksums_sha256.txt").write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

print("Analysis package staged at:")
print(PACKAGE_ROOT)

print("\nTotal files:", len(inventory_df))
print(
    "Total package size:",
    round(inventory_df["size_bytes"].sum() / 1024**2, 2),
    "MB",
)

display(inventory_df.head(20))

Analysis package staged at:
/kaggle/working/skin_lesion_analysis_package

Total files: 139
Total package size: 8.19 MB


,relative_path,extension,size_bytes,size_mb,sha256
0,environment/git_commit.txt,.txt,41,0.0000,e6c7ac5f6841341ff6878c12390fe332abef5bde83b643...
1,environment/git_log.txt,.txt,486,0.0005,a0556c3046bd1963e8971c08b3051fef2eb3a8d3fc72dc...
2,environment/package_versions.txt,.txt,246,0.0002,65b0d11a407b69c2b40a08d57f06a045eae30e43de0f75...
3,environment/python_environment.txt,.txt,291,0.0003,9f9fcd3fa8ac204a6c81f6a75ddaab09477cf186721245...
4,manifest/package_summary.json,.json,699,0.0007,005c3752aadf7394a667a65452977c5a7212a38fb699ec...
5,outputs/figures/.gitkeep,,0,0.0000,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
6,outputs/figures/baseline_risk_coverage_curve_s...,.png,60399,0.0576,859b783b6b7a243f5c3613dabb373d00012bfb9c6ab483...
7,outputs/figures/baseline_uncertainty_correct_v...,.png,40303,0.0384,6db3ebd573ed258f35166ae37eece3ecdfe333e17262b8...
8,outputs/figures/baseline_vs_deep_ensemble_cali...,.png,83882,0.0800,0d117474f237bfee6ba918ecb22d11f9bb6bf95f146dc3...
9,outputs/figures/baseline_vs_deep_ensemble_clas...,.png,88998,0.0849,702b8431e6d1ee601364320287c4881f34f7d62398ae67...


In [139]:
# Phase 10: Compress analysis-ready project package

from pathlib import Path
import shutil

PACKAGE_ROOT = Path("/kaggle/working/skin_lesion_analysis_package")
ZIP_BASE = Path("/kaggle/working/skin_lesion_analysis_package")

analysis_zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=PACKAGE_ROOT.parent,
    base_dir=PACKAGE_ROOT.name,
)

analysis_zip_path = Path(analysis_zip_path)

print("Analysis ZIP created:", analysis_zip_path)
print(
    "Analysis ZIP size:",
    round(analysis_zip_path.stat().st_size / 1024**2, 2),
    "MB",
)

Analysis ZIP created: /kaggle/working/skin_lesion_analysis_package.zip
Analysis ZIP size: 5.0 MB


In [145]:
# Phase 10: Verify analysis ZIP structure and integrity

from pathlib import Path
import zipfile
import pandas as pd

analysis_zip_path = Path(
    "/kaggle/working/skin_lesion_analysis_package.zip"
)

with zipfile.ZipFile(analysis_zip_path, "r") as archive:
    zip_members = archive.namelist()
    corrupt_file = archive.testzip()

print("ZIP exists:", analysis_zip_path.exists())
print("ZIP members:", len(zip_members))
print("Corrupt member:", corrupt_file)

required_zip_entries = [
    "skin_lesion_analysis_package/project_metadata/README.md",
    "skin_lesion_analysis_package/project_metadata/configs/experiment.yaml",
    "skin_lesion_analysis_package/source_snapshot/src/",
    "skin_lesion_analysis_package/outputs/tables/final_model_comparison.csv",
    "skin_lesion_analysis_package/outputs/tables/final_uncertainty_comparison.csv",
    "skin_lesion_analysis_package/outputs/tables/final_risk_coverage_comparison.csv",
    "skin_lesion_analysis_package/outputs/figures/final_classification_comparison.png",
    "skin_lesion_analysis_package/outputs/figures/final_calibration_comparison.png",
    "skin_lesion_analysis_package/outputs/figures/final_risk_coverage_comparison.png",
    "skin_lesion_analysis_package/manifest/file_inventory.csv",
    "skin_lesion_analysis_package/manifest/checksums_sha256.txt",
    "skin_lesion_analysis_package/manifest/package_summary.json",
]

zip_verification_rows = []

for required_entry in required_zip_entries:
    if required_entry.endswith("/"):
        exists = any(
            member.startswith(required_entry)
            for member in zip_members
        )
    else:
        exists = required_entry in zip_members

    zip_verification_rows.append({
        "required_entry": required_entry,
        "exists": exists,
    })

zip_verification_df = pd.DataFrame(zip_verification_rows)
display(zip_verification_df)

zip_ready = (
    analysis_zip_path.exists()
    and analysis_zip_path.stat().st_size > 0
    and corrupt_file is None
    and zip_verification_df["exists"].all()
)

print("\nAnalysis ZIP ready for upload:", zip_ready)

ZIP exists: True
ZIP members: 154
Corrupt member: None


,required_entry,exists
0,skin_lesion_analysis_package/project_metadata/...,True
1,skin_lesion_analysis_package/project_metadata/...,True
2,skin_lesion_analysis_package/source_snapshot/src/,True
3,skin_lesion_analysis_package/outputs/tables/fi...,True
4,skin_lesion_analysis_package/outputs/tables/fi...,True
5,skin_lesion_analysis_package/outputs/tables/fi...,True
6,skin_lesion_analysis_package/outputs/figures/f...,True
7,skin_lesion_analysis_package/outputs/figures/f...,True
8,skin_lesion_analysis_package/outputs/figures/f...,True
9,skin_lesion_analysis_package/manifest/file_inv...,True



Analysis ZIP ready for upload: True


In [141]:
# Phase 10: Build complete private project archive with checkpoints

from pathlib import Path
from datetime import datetime
import shutil

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
FULL_PACKAGE_ROOT = Path(
    "/kaggle/working/skin_lesion_complete_archive"
)

if FULL_PACKAGE_ROOT.exists():
    shutil.rmtree(FULL_PACKAGE_ROOT)

FULL_PACKAGE_ROOT.mkdir(parents=True)

# Copy important repository files.
for item_name in [
    "README.md",
    ".gitignore",
    "requirements.txt",
    "requirements-local.txt",
    "configs",
    "src",
    "tests",
    "outputs",
]:
    source_path = REPO_DIR / item_name
    destination_path = FULL_PACKAGE_ROOT / item_name

    if source_path.is_dir():
        shutil.copytree(
            source_path,
            destination_path,
            dirs_exist_ok=True,
            ignore=shutil.ignore_patterns(
                "__pycache__",
                "*.pyc",
                ".pytest_cache",
                "*.zip",
            ),
        )
    elif source_path.exists():
        shutil.copy2(
            source_path,
            destination_path,
        )

full_zip_path = shutil.make_archive(
    base_name=str(
        Path("/kaggle/working")
        / "skin_lesion_complete_archive"
    ),
    format="zip",
    root_dir=FULL_PACKAGE_ROOT.parent,
    base_dir=FULL_PACKAGE_ROOT.name,
)

full_zip_path = Path(full_zip_path)

print("Complete archive created:", full_zip_path)
print(
    "Complete archive size:",
    round(full_zip_path.stat().st_size / 1024**2, 2),
    "MB",
)

Complete archive created: /kaggle/working/skin_lesion_complete_archive.zip
Complete archive size: 132.73 MB


In [142]:
# Phase 10: Create checkpoint-only backup ZIP

from pathlib import Path
import shutil

REPO_DIR = Path("/kaggle/working/skin-lesion-uncertainty")
CHECKPOINTS_DIR = REPO_DIR / "outputs/checkpoints"

checkpoint_zip_path = shutil.make_archive(
    base_name=str(
        Path("/kaggle/working")
        / "skin_lesion_model_checkpoints"
    ),
    format="zip",
    root_dir=CHECKPOINTS_DIR.parent,
    base_dir=CHECKPOINTS_DIR.name,
)

checkpoint_zip_path = Path(checkpoint_zip_path)

print("Checkpoint ZIP created:", checkpoint_zip_path)
print(
    "Checkpoint ZIP size:",
    round(checkpoint_zip_path.stat().st_size / 1024**2, 2),
    "MB",
)

Checkpoint ZIP created: /kaggle/working/skin_lesion_model_checkpoints.zip
Checkpoint ZIP size: 127.74 MB
